In [1]:
# %pip install seaborn
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
import warnings
warnings.filterwarnings('ignore')
import re
from math import radians, sin, cos, sqrt, atan2
# 设置中文字体显示（如果图表需要显示中文）
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

In [2]:
# 读取Excel文件
try:
    # 读取租房数据  
    rent_price_df = pd.read_excel(r"D:\desktop\ruc_Class25Q2_test_rent.xlsx")
    print("租房数据读取成功！")
    print(f"租房数据形状: {rent_price_df.shape}")
    
except FileNotFoundError as e:
    print(f"文件未找到: {e}")
except Exception as e:
    print(f"读取文件时出错: {e}")

# 查看数据的基本信息
print("\n=== 租房数据基本信息 ===")
print(f"行数: {rent_price_df.shape[0]}, 列数: {rent_price_df.shape[1]}")
print(rent_price_df.info())

# 查看前几行数据
print("\n=== 租房数据前3行 ===")
print(rent_price_df.head(3))

# 检查列名
print("\n=== 租房数据列名 ===")
print(rent_price_df.columns.tolist())

租房数据读取成功！
租房数据形状: (9773, 46)

=== 租房数据基本信息 ===
行数: 9773, 列数: 46
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9773 entries, 0 to 9772
Data columns (total 46 columns):
 #   Column   Non-Null Count  Dtype         
---  ------   --------------  -----         
 0   ID       9773 non-null   int64         
 1   城市       9773 non-null   int64         
 2   户型       9773 non-null   object        
 3   装修       7091 non-null   object        
 4   楼层       9773 non-null   object        
 5   面积       9773 non-null   object        
 6   朝向       9773 non-null   object        
 7   交易时间     9773 non-null   datetime64[ns]
 8   付款方式     7386 non-null   object        
 9   租赁方式     9773 non-null   object        
 10  电梯       9773 non-null   object        
 11  车位       2162 non-null   object        
 12  用水       7500 non-null   object        
 13  用电       7546 non-null   object        
 14  燃气       9182 non-null   object        
 15  采暖       3213 non-null   object        
 16  租期       4598 

In [3]:
# 2.租房数据-检查各个变量的分布

# 2.1 装修变量
# 2.1.1 空缺值检查
# 因为初步判断为0-1变量，且空缺值很多，只检查所含种类
print(rent_price_df['装修'].value_counts())

# 2.1.2 变量类型转化
# 验证后确实是0-1变量，故将包含"精装修"的设为1，其他设为0
rent_price_df['decoration'] = rent_price_df['装修'].astype(str).str.contains('精装修').astype(int)
# 验证转换结果
print("转换后的装修列分布:")
print(rent_price_df['decoration'].value_counts())

装修
精装修    7091
Name: count, dtype: int64
转换后的装修列分布:
decoration
1    7091
0    2682
Name: count, dtype: int64


In [4]:
# 2.2 楼层
# 2.2.1 分布检查
print("楼层分布:")
print(rent_price_df['楼层'].value_counts())

# 2.2.2 缺失值检查
print(f"\n楼层缺失值数量: {rent_price_df['楼层'].isna().sum()}")

# 2.2.3 创建楼层变量
# 删除已存在的楼层变量列
floor_columns_to_drop = [col for col in rent_price_df.columns if 
                        col in ['high_dummy', 'middle_dummy', 'low_dummy', 
                               'basement_dummy', 'total_floor']]
rent_price_df = rent_price_df.drop(columns=floor_columns_to_drop)

# 从楼层数据中提取5个变量:
# 1. 高哑变量 (high_dummy)
# 2. 中哑变量 (middle_dummy) 
# 3. 低哑变量 (low_dummy)
# 4. 地下室哑变量 (basement_dummy)
# 5. 总楼层连续变量 (total_floor)
# 保留缺失值为NaN，不进行填充

def determine_floor_level(current_floor, total_floor):
    """
    根据当前楼层和总楼层判断楼层类型
    使用3n, 3n+1, 3n+2的逻辑
    """
    if current_floor <= 0 or total_floor <= 0:
        return None
    
    # 计算n值
    if total_floor % 3 == 0:  # 3n情况
        n = total_floor // 3
        if current_floor <= n:
            return 'low'
        elif current_floor <= 2 * n:
            return 'middle'
        else:
            return 'high'
    
    elif total_floor % 3 == 1:  # 3n+1情况
        n = (total_floor - 1) // 3
        if current_floor <= n:
            return 'low'
        elif current_floor <= total_floor - n:
            return 'middle'
        else:
            return 'high'
    
    else:  # 3n+2情况
        n = (total_floor + 1) // 3
        if current_floor <= n:
            return 'low'
        elif current_floor <= total_floor - n:
            return 'middle'
        else:
            return 'high'

def extract_floor_features(df, floor_column='楼层'):
    """
    从楼层数据中提取5个变量:
    1. 高哑变量 (high_dummy)
    2. 中哑变量 (middle_dummy) 
    3. 低哑变量 (low_dummy)
    4. 地下室哑变量 (basement_dummy)
    5. 总楼层连续变量 (total_floor)
    """
    # 创建新列，初始化为NaN
    df['high_dummy'] = np.nan
    df['middle_dummy'] = np.nan
    df['low_dummy'] = np.nan
    df['basement_dummy'] = np.nan
    df['total_floor'] = np.nan
    
    # 只对非空楼层进行处理
    valid_mask = df[floor_column].notna()
    
    # 将楼层列转换为字符串类型
    floor_str = df.loc[valid_mask, floor_column].astype(str)
    
    # 初始化有效数据中的哑变量为0
    df.loc[valid_mask, 'high_dummy'] = 0
    df.loc[valid_mask, 'middle_dummy'] = 0
    df.loc[valid_mask, 'low_dummy'] = 0
    df.loc[valid_mask, 'basement_dummy'] = 0
    
    # 处理地下室情况
    basement_mask = floor_str.str.contains('地下室')
    df.loc[valid_mask & basement_mask, 'basement_dummy'] = 1
    # 地下室的总楼层设为0
    df.loc[valid_mask & basement_mask, 'total_floor'] = 0
    
    # 处理高楼层情况（直接标注的）
    high_mask = floor_str.str.contains('高楼层')
    df.loc[valid_mask & high_mask, 'high_dummy'] = 1
    
    # 处理中楼层情况（直接标注的）
    middle_mask = floor_str.str.contains('中楼层')
    df.loc[valid_mask & middle_mask, 'middle_dummy'] = 1
    
    # 处理低楼层情况（直接标注的）
    low_mask = floor_str.str.contains('低楼层')
    df.loc[valid_mask & low_mask, 'low_dummy'] = 1
    
    # 处理"XX/XX层"格式，提取当前楼层和总楼层
    pattern_mask = floor_str.str.match(r'^\d+/\d+层?$')
    if pattern_mask.any():
        # 分割字符串并提取数字
        split_data = floor_str[pattern_mask].str.split('/', expand=True)
        current_floors = pd.to_numeric(split_data[0], errors='coerce')
        total_floors = pd.to_numeric(split_data[1].str.extract(r'(\d+)')[0], errors='coerce')
        
        # 设置总楼层
        df.loc[valid_mask & pattern_mask, 'total_floor'] = total_floors
        
        # 对于"XX/XX层"格式且没有被直接标注为高中低楼层的数据，使用逻辑判断楼层类型
        unclassified_pattern_mask = pattern_mask & ~(high_mask | middle_mask | low_mask)
        if unclassified_pattern_mask.any():
            for idx in df[valid_mask & unclassified_pattern_mask].index:
                current_floor = current_floors.loc[idx]
                total_floor = total_floors.loc[idx]
                
                # 使用逻辑判断楼层类型
                floor_level = determine_floor_level(current_floor, total_floor)
                if floor_level == 'high':
                    df.loc[idx, 'high_dummy'] = 1
                elif floor_level == 'middle':
                    df.loc[idx, 'middle_dummy'] = 1
                elif floor_level == 'low':
                    df.loc[idx, 'low_dummy'] = 1
    
    # 处理"高/中/低楼层/XX层"格式，提取总楼层
    level_mask = (high_mask | middle_mask | low_mask) & ~basement_mask
    if level_mask.any():
        df.loc[valid_mask & level_mask, 'total_floor'] = pd.to_numeric(
            floor_str[level_mask].str.extract(r'/(\d+)层')[0], errors='coerce'
        )
    
    # 确保哑变量是互斥的（一个楼层只能属于一种类型）
    # 如果有同时标记为多种类型的情况，我们按优先级处理：地下室 > 高楼层 > 中楼层 > 低楼层
    conflict_mask = (
        (df['basement_dummy'] == 1) & 
        ((df['high_dummy'] == 1) | (df['middle_dummy'] == 1) | (df['low_dummy'] == 1))
    )
    if conflict_mask.any():
        df.loc[conflict_mask, 'high_dummy'] = 0
        df.loc[conflict_mask, 'middle_dummy'] = 0
        df.loc[conflict_mask, 'low_dummy'] = 0
    
    conflict_mask = (
        (df['high_dummy'] == 1) & 
        ((df['middle_dummy'] == 1) | (df['low_dummy'] == 1))
    )
    if conflict_mask.any():
        df.loc[conflict_mask, 'middle_dummy'] = 0
        df.loc[conflict_mask, 'low_dummy'] = 0
    
    conflict_mask = (df['middle_dummy'] == 1) & (df['low_dummy'] == 1)
    if conflict_mask.any():
        df.loc[conflict_mask, 'low_dummy'] = 0
    
    return df

# 应用函数提取楼层特征
rent_price_df = extract_floor_features(rent_price_df, '楼层')

print(f"\n已创建的楼层变量:")
print(f"high_dummy: 1表示高楼层，0表示非高楼层，NaN表示缺失")
print(f"middle_dummy: 1表示中楼层，0表示非中楼层，NaN表示缺失")
print(f"low_dummy: 1表示低楼层，0表示非低楼层，NaN表示缺失")
print(f"basement_dummy: 1表示地下室，0表示非地下室，NaN表示缺失")
print(f"total_floor: 总楼层数，NaN表示缺失")

print("\n楼层变量统计:")
floor_features = ['high_dummy', 'middle_dummy', 'low_dummy', 'basement_dummy', 'total_floor']
for feature in floor_features:
    if feature == 'total_floor':
        print(f"{feature}:")
        print(f"  非缺失值数量: {rent_price_df[feature].notna().sum()}")
        if rent_price_df[feature].notna().sum() > 0:
            print(f"  平均值: {rent_price_df[feature].mean():.2f}")
            print(f"  标准差: {rent_price_df[feature].std():.2f}")
            print(f"  最小值: {rent_price_df[feature].min()}")
            print(f"  最大值: {rent_price_df[feature].max()}")
        print(f"  缺失值数量: {rent_price_df[feature].isna().sum()}")
    else:
        print(f"{feature}:")
        print(f"  为1的数量: {(rent_price_df[feature] == 1).sum()}")
        print(f"  为0的数量: {(rent_price_df[feature] == 0).sum()}")
        print(f"  缺失值数量: {rent_price_df[feature].isna().sum()}")

print("楼层变量创建完成（保留缺失值）")

楼层分布:
楼层
高楼层/6层     508
中楼层/6层     327
高楼层/18层    239
低楼层/6层     226
低楼层/18层    219
          ... 
21/23层       1
1/16层        1
17/32层       1
5/9层         1
11/30层       1
Name: count, Length: 613, dtype: int64

楼层缺失值数量: 0

已创建的楼层变量:
high_dummy: 1表示高楼层，0表示非高楼层，NaN表示缺失
middle_dummy: 1表示中楼层，0表示非中楼层，NaN表示缺失
low_dummy: 1表示低楼层，0表示非低楼层，NaN表示缺失
basement_dummy: 1表示地下室，0表示非地下室，NaN表示缺失
total_floor: 总楼层数，NaN表示缺失

楼层变量统计:
high_dummy:
  为1的数量: 3282
  为0的数量: 6491
  缺失值数量: 0
middle_dummy:
  为1的数量: 3629
  为0的数量: 6144
  缺失值数量: 0
low_dummy:
  为1的数量: 2822
  为0的数量: 6951
  缺失值数量: 0
basement_dummy:
  为1的数量: 40
  为0的数量: 9733
  缺失值数量: 0
total_floor:
  非缺失值数量: 9773
  平均值: 18.96
  标准差: 11.15
  最小值: 0.0
  最大值: 58.0
  缺失值数量: 0
楼层变量创建完成（保留缺失值）


In [5]:
# 2.3 面积变量
# 2.3.1 空缺值（在前面的描述性统计中已看出，没有空缺值）
# 2.3.2 变量类型转化
rent_price_df['area'] = rent_price_df['面积'].str.replace('㎡', '').astype(float)
print("面积提取完成!")

面积提取完成!


In [6]:
# 2.4 户型
# 2.4.1 分布检查
print("户型分布:")
print(rent_price_df['户型'].value_counts())

# 2.4.2 缺失值检查
print(f"\n户型缺失值数量: {rent_price_df['户型'].isna().sum()}")

# 2.4.3 创建户型变量
# 删除已存在的户型变量列
layout_columns_to_drop = [col for col in rent_price_df.columns if 
                         col in ['room_count', 'hall_count']]
rent_price_df = rent_price_df.drop(columns=layout_columns_to_drop)

# 从户型数据中提取2个变量:
# 1. 室数量 (room_count)
# 2. 厅数量 (hall_count)
# 保留缺失值为NaN，不进行填充

def extract_room_features(df, layout_column='户型'):
    """
    从户型数据中提取2个变量:
    1. 室数量 (room_count)
    2. 厅数量 (hall_count)
    
    处理逻辑:
    - 提取"室"前面的数字作为室数量
    - 提取"厅"前面的数字作为厅数量  
    - 将"房间"、"居室"视为"室"
    - 无法识别的设为NaN
    """
    # 创建新列，初始化为NaN
    df['room_count'] = np.nan
    df['hall_count'] = np.nan
    
    # 只对非空户型进行处理
    valid_mask = df[layout_column].notna()
    
    # 将户型列转换为字符串类型
    layout_str = df.loc[valid_mask, layout_column].astype(str)
    
    # 提取室数量
    # 匹配"数字+室"、"数字+房间"、"数字+居室"
    room_pattern = r'(\d+)(?:室|房间|居室)'
    room_matches = layout_str.str.extract(room_pattern, expand=False)
    df.loc[valid_mask, 'room_count'] = pd.to_numeric(room_matches, errors='coerce')
    
    # 提取厅数量
    # 匹配"数字+厅"
    hall_pattern = r'(\d+)厅'
    hall_matches = layout_str.str.extract(hall_pattern, expand=False)
    df.loc[valid_mask, 'hall_count'] = pd.to_numeric(hall_matches, errors='coerce')
    
    # 特殊处理：对于"未知室"等情况，设为NaN
    unknown_mask = layout_str.str.contains('未知室')
    df.loc[valid_mask & unknown_mask, 'room_count'] = np.nan
    
    # 特殊处理：对于"车库"等情况，设为NaN
    garage_mask = layout_str.str.contains('车库')
    df.loc[valid_mask & garage_mask, 'room_count'] = np.nan
    df.loc[valid_mask & garage_mask, 'hall_count'] = np.nan
    
    return df

# 应用函数提取户型特征
rent_price_df = extract_room_features(rent_price_df, '户型')


print("\n户型变量统计:")
print("room_count:")
print(f"  非缺失值数量: {rent_price_df['room_count'].notna().sum()}")
print(f"  缺失值数量: {rent_price_df['room_count'].isna().sum()}")
if rent_price_df['room_count'].notna().sum() > 0:
    print(f"  平均值: {rent_price_df['room_count'].mean():.2f}")
    print(f"  标准差: {rent_price_df['room_count'].std():.2f}")
    print(f"  最小值: {rent_price_df['room_count'].min()}")
    print(f"  最大值: {rent_price_df['room_count'].max()}")

print("\nhall_count:")
print(f"  非缺失值数量: {rent_price_df['hall_count'].notna().sum()}")
print(f"  缺失值数量: {rent_price_df['hall_count'].isna().sum()}")
if rent_price_df['hall_count'].notna().sum() > 0:
    print(f"  平均值: {rent_price_df['hall_count'].mean():.2f}")
    print(f"  标准差: {rent_price_df['hall_count'].std():.2f}")
    print(f"  最小值: {rent_price_df['hall_count'].min()}")
    print(f"  最大值: {rent_price_df['hall_count'].max()}")

print("户型变量创建完成（保留缺失值）")

户型分布:
户型
2室1厅1卫    2046
1室1厅1卫    1673
3室2厅2卫    1279
2室2厅1卫    1051
3室2厅1卫     912
          ... 
6室3厅2卫       1
5室2厅5卫       1
3室0厅0卫       1
7房间2卫        1
4室4厅4卫       1
Name: count, Length: 97, dtype: int64

户型缺失值数量: 0

户型变量统计:
room_count:
  非缺失值数量: 9761
  缺失值数量: 12
  平均值: 2.35
  标准差: 1.03
  最小值: 1.0
  最大值: 10.0

hall_count:
  非缺失值数量: 9644
  缺失值数量: 129
  平均值: 1.38
  标准差: 0.59
  最小值: 0.0
  最大值: 5.0
户型变量创建完成（保留缺失值）


In [7]:
# 2.5 朝向
# 2.5.1 分布检查
print("朝向分布:")
print(rent_price_df['朝向'].value_counts())

# 2.5.2 缺失值检查
print(f"\n朝向缺失值数量: {rent_price_df['朝向'].isna().sum()}")

# 2.5.3 创建朝向哑变量
# 删除已存在的朝向哑变量列
orientation_columns_to_drop = [col for col in rent_price_df.columns if 
                              col in ['south_dummy', 'north_south_dummy']]
rent_price_df = rent_price_df.drop(columns=orientation_columns_to_drop)

# 从朝向数据中提取2个哑变量:
# 1. 朝南哑变量 (south_dummy) - 只要包含"南"字就算朝南
# 2. 南北通透哑变量 (north_south_dummy) - 同时包含"南"和"北"字
# 保留缺失值为NaN，不进行填充

# 初始化哑变量为NaN
rent_price_df['south_dummy'] = np.nan
rent_price_df['north_south_dummy'] = np.nan

# 只对非空朝向进行处理
valid_mask = rent_price_df['朝向'].notna()

# 将朝向列转换为字符串类型
orientation_str = rent_price_df.loc[valid_mask, '朝向'].astype(str)

# 提取朝南哑变量 - 只要包含"南"字就算朝南
south_mask = orientation_str.str.contains('南')
rent_price_df.loc[valid_mask & south_mask, 'south_dummy'] = 1
rent_price_df.loc[valid_mask & ~south_mask, 'south_dummy'] = 0

# 提取南北通透哑变量 - 同时包含"南"和"北"字
north_south_mask = orientation_str.str.contains('南') & orientation_str.str.contains('北')
rent_price_df.loc[valid_mask & north_south_mask, 'north_south_dummy'] = 1
rent_price_df.loc[valid_mask & ~north_south_mask, 'north_south_dummy'] = 0

print(f"\n已创建的朝向哑变量:")
print(f"south_dummy: 1表示朝南，0表示不朝南，NaN表示缺失")
print(f"north_south_dummy: 1表示南北通透，0表示非南北通透，NaN表示缺失")

print("\n朝向哑变量分布:")
print(f"朝南 (south_dummy=1): {(rent_price_df['south_dummy'] == 1).sum()}")
print(f"不朝南 (south_dummy=0): {(rent_price_df['south_dummy'] == 0).sum()}")
print(f"朝南缺失值: {rent_price_df['south_dummy'].isna().sum()}")

print(f"\n南北通透 (north_south_dummy=1): {(rent_price_df['north_south_dummy'] == 1).sum()}")
print(f"非南北通透 (north_south_dummy=0): {(rent_price_df['north_south_dummy'] == 0).sum()}")
print(f"南北通透缺失值: {rent_price_df['north_south_dummy'].isna().sum()}")

print("朝向哑变量创建完成（保留缺失值）")

朝向分布:
朝向
南               4651
南 北             1759
东南               892
北                628
东                594
西南               301
西                265
东北               135
西北               116
东 西               88
东南 南              64
东 南               39
东 东南              29
南 西南              28
东 北               16
南 西               15
东南 北              13
南 西 北             13
东南 西南             12
西 北               11
西南 北              11
南 西北              10
东 南 北             10
东南 西北              9
西南 西               7
北 东北               6
东 东北               4
西南 东北              3
东南 东北              3
西 西北               3
东南 南 北             3
东南 南 西南            3
西北 北               3
东 东南 南             3
东 南 西              2
未知                 2
东 南 西 北            2
西 西北 北             2
东南 西               1
东 东南 南 北           1
南 西 西北             1
南 东北               1
西 北 东北             1
东 西南               1
东 南 西南             1
南 西南 北             1
东 东南 南 西南          1
东 西北

In [8]:
# 2.6 交易时间变量
# 2.6.1 异常值检查以及处理
null_index = rent_price_df[rent_price_df['交易时间'].isnull()].index
print("空缺行的索引:", null_index.tolist())

# 2.6.2 具体内容
print("交易时间分布:")
print(rent_price_df['交易时间'].value_counts().head(20))

# 2.6.3 提取年份和季度共8个哑变量
# 首先确保交易时间是日期格式
rent_price_df['交易时间'] = pd.to_datetime(rent_price_df['交易时间'], errors='coerce')

# 检查转换后的日期范围
print(f"\n交易时间转换后的日期范围:")
print(f"最早日期: {rent_price_df['交易时间'].min()}")
print(f"最晚日期: {rent_price_df['交易时间'].max()}")
print(f"交易时间缺失值数量: {rent_price_df['交易时间'].isna().sum()}")

# 提取年份和季度
rent_price_df['交易年份'] = rent_price_df['交易时间'].dt.year
rent_price_df['交易季度'] = rent_price_df['交易时间'].dt.quarter

# 检查年份和季度分布
print(f"\n交易年份分布:")
print(rent_price_df['交易年份'].value_counts().sort_index())
print(f"\n交易季度分布:")
print(rent_price_df['交易季度'].value_counts().sort_index())

# 删除已存在的交易时间哑变量列
transaction_columns_to_drop = [col for col in rent_price_df.columns if col.startswith('trans_')]
rent_price_df = rent_price_df.drop(columns=transaction_columns_to_drop)

# 创建年份-季度组合列
rent_price_df['交易年季度'] = rent_price_df['交易年份'].astype(str) + 'Q' + rent_price_df['交易季度'].astype(str)

# 检查交易年季度分布
print(f"\n交易年季度分布:")
print(rent_price_df['交易年季度'].value_counts().sort_index())

# 创建8个季度哑变量（2024年Q1-Q4，2025年Q1-Q4）
# 定义所有可能的季度
all_quarters = [f"{year}Q{quarter}" for year in [2024, 2025] for quarter in range(1, 5)]

# 创建哑变量
transaction_dummies = pd.get_dummies(rent_price_df['交易年季度'], prefix='trans')

# 确保包含所有8个季度（即使某些季度没有数据）
for quarter in all_quarters:
    col_name = f"trans_{quarter}"
    if col_name not in transaction_dummies.columns:
        transaction_dummies[col_name] = 0

# 按正确的顺序排列列
transaction_dummies = transaction_dummies.reindex(columns=[f"trans_{quarter}" for quarter in all_quarters])

# 将布尔值转换为整数 (True/False -> 1/0)
transaction_dummies = transaction_dummies.astype(int)

# 将原始数据中的缺失值反映到哑变量中
if rent_price_df['交易时间'].isna().any():
    na_mask = rent_price_df['交易时间'].isna()
    for col in transaction_dummies.columns:
        transaction_dummies.loc[na_mask, col] = np.nan

# 将哑变量合并到原数据框
rent_price_df = pd.concat([rent_price_df, transaction_dummies], axis=1)

# 删除中间列
rent_price_df = rent_price_df.drop(['交易年份', '交易季度', '交易年季度'], axis=1)

print("\n已创建的交易时间季度哑变量列:")
transaction_dummy_columns = [col for col in rent_price_df.columns if col.startswith('trans_')]
print(transaction_dummy_columns)

print("\n交易时间季度哑变量分布:")
for col in transaction_dummy_columns:
    valid_count = rent_price_df[col].notna().sum()
    true_count = (rent_price_df[col] == 1).sum()
    na_count = rent_price_df[col].isna().sum()
    print(f"{col}: {true_count}个为1, {valid_count-true_count}个为0, {na_count}个缺失")

print("交易时间季度哑变量创建完成（保留缺失值）")

空缺行的索引: []
交易时间分布:
交易时间
2025-06-10    84
2025-06-19    82
2025-06-11    82
2025-05-11    82
2025-03-20    81
2025-04-23    80
2025-03-23    79
2025-03-11    77
2025-05-04    76
2025-04-22    76
2025-03-31    76
2025-06-12    75
2025-03-21    75
2025-03-06    74
2025-03-14    74
2025-05-02    73
2025-03-12    73
2025-02-16    72
2025-03-25    70
2025-04-09    70
Name: count, dtype: int64

交易时间转换后的日期范围:
最早日期: 2025-02-14 00:00:00
最晚日期: 2025-11-12 00:00:00
交易时间缺失值数量: 0

交易年份分布:
交易年份
2025    9773
Name: count, dtype: int64

交易季度分布:
交易季度
1    2867
2    4092
3    2309
4     505
Name: count, dtype: int64

交易年季度分布:
交易年季度
2025Q1    2867
2025Q2    4092
2025Q3    2309
2025Q4     505
Name: count, dtype: int64

已创建的交易时间季度哑变量列:
['trans_2024Q1', 'trans_2024Q2', 'trans_2024Q3', 'trans_2024Q4', 'trans_2025Q1', 'trans_2025Q2', 'trans_2025Q3', 'trans_2025Q4']

交易时间季度哑变量分布:
trans_2024Q1: 0个为1, 9773个为0, 0个缺失
trans_2024Q2: 0个为1, 9773个为0, 0个缺失
trans_2024Q3: 0个为1, 9773个为0, 0个缺失
trans_2024Q4: 0个为1, 9773个为0, 0个缺失

In [9]:
# 2.7 付款方式
# 2.7.1 分布检查
print("付款方式分布:")
print(rent_price_df['付款方式'].value_counts())

# 2.7.2 异常值处理
# 将包含链接的付款方式设为NaN
link_mask = rent_price_df['付款方式'].str.contains('http', na=False)
rent_price_df.loc[link_mask, '付款方式'] = np.nan
print(f"\n已将 {link_mask.sum()} 个包含链接的付款方式设为空值")

# 检查缺失值情况
print(f"\n付款方式缺失值数量: {rent_price_df['付款方式'].isna().sum()}")

# 2.7.3 创建付款方式哑变量
# 删除已存在的付款方式哑变量列
payment_columns_to_drop = [col for col in rent_price_df.columns if col.startswith('pay_')]
rent_price_df = rent_price_df.drop(columns=payment_columns_to_drop)

# 创建付款方式到英文的映射字典
payment_mapping = {
    '季付价': 'quarterly',
    '月付价': 'monthly', 
    '半年付价': 'semi_annual',
    '年付价': 'annual',
    '双月付价': 'bi_monthly'
}

# 将付款方式转换为英文并创建哑变量
rent_price_df['payment_method_en'] = rent_price_df['付款方式'].map(payment_mapping)
payment_dummies = pd.get_dummies(rent_price_df['payment_method_en'], prefix='pay')

# 将布尔值转换为整数 (True/False -> 1/0)
payment_dummies = payment_dummies.astype(int)

# 将原始数据中的缺失值反映到哑变量中
if rent_price_df['付款方式'].isna().any():
    na_mask = rent_price_df['付款方式'].isna()
    for col in payment_dummies.columns:
        payment_dummies.loc[na_mask, col] = np.nan

# 将哑变量合并到原数据框并删除中间列
rent_price_df = pd.concat([rent_price_df, payment_dummies], axis=1)
rent_price_df = rent_price_df.drop('payment_method_en', axis=1)

print(f"\n已创建的付款方式哑变量:")
payment_dummy_columns = [col for col in rent_price_df.columns if col.startswith('pay_')]
print(payment_dummy_columns)

print("\n付款方式哑变量分布:")
for col in payment_dummy_columns:
    valid_count = rent_price_df[col].notna().sum()
    true_count = (rent_price_df[col] == 1).sum()
    na_count = rent_price_df[col].isna().sum()
    print(f"{col}: {true_count}个为1, {valid_count-true_count}个为0, {na_count}个缺失")

print("付款方式哑变量创建完成（保留缺失值）")

付款方式分布:
付款方式
季付价     5325
月付价     1726
半年付价     284
年付价       49
双月付价       2
Name: count, dtype: int64

已将 0 个包含链接的付款方式设为空值

付款方式缺失值数量: 2387

已创建的付款方式哑变量:
['pay_annual', 'pay_bi_monthly', 'pay_monthly', 'pay_quarterly', 'pay_semi_annual']

付款方式哑变量分布:
pay_annual: 49个为1, 7337个为0, 2387个缺失
pay_bi_monthly: 2个为1, 7384个为0, 2387个缺失
pay_monthly: 1726个为1, 5660个为0, 2387个缺失
pay_quarterly: 5325个为1, 2061个为0, 2387个缺失
pay_semi_annual: 284个为1, 7102个为0, 2387个缺失
付款方式哑变量创建完成（保留缺失值）


In [10]:
# 2.8 租赁方式
# 2.8.1 分布检查
print("租赁方式分布:")
print(rent_price_df['租赁方式'].value_counts())

# 2.8.2 异常值处理
# 检查是否有异常值或缺失值
print(f"\n租赁方式缺失值数量: {rent_price_df['租赁方式'].isna().sum()}")

# 2.8.3 创建租赁方式哑变量
# 删除已存在的租赁方式哑变量列
rent_type_columns_to_drop = [col for col in rent_price_df.columns if col.startswith('rent_type_')]
rent_price_df = rent_price_df.drop(columns=rent_type_columns_to_drop)

# 创建租赁方式哑变量
# 以"整租"为基准，创建"合租"的哑变量
# 保留缺失值为NaN，不进行填充
rent_price_df['rent_type_shared'] = rent_price_df['租赁方式'].map({'整租': 0, '合租': 1})

print(f"\n已创建的租赁方式哑变量:")
print(f"rent_type_shared: 1表示合租，0表示整租，NaN表示缺失")

print("\n租赁方式哑变量分布:")
print(f"整租 (rent_type_shared=0): {(rent_price_df['rent_type_shared'] == 0).sum()}")
print(f"合租 (rent_type_shared=1): {(rent_price_df['rent_type_shared'] == 1).sum()}")
print(f"缺失值: {rent_price_df['rent_type_shared'].isna().sum()}")

print("租赁方式哑变量创建完成（保留缺失值）")

租赁方式分布:
租赁方式
整租    9128
合租     645
Name: count, dtype: int64

租赁方式缺失值数量: 0

已创建的租赁方式哑变量:
rent_type_shared: 1表示合租，0表示整租，NaN表示缺失

租赁方式哑变量分布:
整租 (rent_type_shared=0): 9128
合租 (rent_type_shared=1): 645
缺失值: 0
租赁方式哑变量创建完成（保留缺失值）


In [11]:
# 2.9 电梯
# 2.9.1 分布检查
print("电梯分布:")
print(rent_price_df['电梯'].value_counts())

# 2.9.2 异常值处理
# 检查是否有异常值或缺失值
print(f"\n电梯缺失值数量: {rent_price_df['电梯'].isna().sum()}")

# 2.9.3 创建电梯哑变量
# 删除已存在的电梯哑变量列
elevator_columns_to_drop = [col for col in rent_price_df.columns if col.startswith('elevator_')]
rent_price_df = rent_price_df.drop(columns=elevator_columns_to_drop)

# 创建电梯哑变量
# 以"无电梯"为基准，创建"有电梯"的哑变量
# 保留缺失值为NaN，不进行填充
rent_price_df['elevator_yes'] = rent_price_df['电梯'].map({'无': 0, '有': 1})

print(f"\n已创建的电梯哑变量:")
print(f"elevator_yes: 1表示有电梯，0表示无电梯，NaN表示缺失")

print("\n电梯哑变量分布:")
print(f"无电梯 (elevator_yes=0): {(rent_price_df['elevator_yes'] == 0).sum()}")
print(f"有电梯 (elevator_yes=1): {(rent_price_df['elevator_yes'] == 1).sum()}")
print(f"缺失值: {rent_price_df['elevator_yes'].isna().sum()}")

print("电梯哑变量创建完成（保留缺失值）")

电梯分布:
电梯
有    6496
无    3277
Name: count, dtype: int64

电梯缺失值数量: 0

已创建的电梯哑变量:
elevator_yes: 1表示有电梯，0表示无电梯，NaN表示缺失

电梯哑变量分布:
无电梯 (elevator_yes=0): 3277
有电梯 (elevator_yes=1): 6496
缺失值: 0
电梯哑变量创建完成（保留缺失值）


In [12]:
# 2.10 用水用电燃气
# 2.10.1 分布检查
print("用水分布:")
print(rent_price_df['用水'].value_counts())

print("\n用电分布:")
print(rent_price_df['用电'].value_counts())

print("\n燃气分布:")
print(rent_price_df['燃气'].value_counts())

print("\n供水分布:")
print(rent_price_df['供水'].value_counts().head(10))  # 只显示前10个

print("\n供电分布:")
print(rent_price_df['供电'].value_counts().head(10))  # 只显示前10个

print("\n燃气费分布:")
print(rent_price_df['燃气费'].value_counts().head(10))  # 只显示前10个

# 2.10.2 缺失值检查
print(f"\n用水缺失值数量: {rent_price_df['用水'].isna().sum()}")
print(f"用电缺失值数量: {rent_price_df['用电'].isna().sum()}")
print(f"燃气缺失值数量: {rent_price_df['燃气'].isna().sum()}")
print(f"供水缺失值数量: {rent_price_df['供水'].isna().sum()}")
print(f"供电缺失值数量: {rent_price_df['供电'].isna().sum()}")
print(f"燃气费缺失值数量: {rent_price_df['燃气费'].isna().sum()}")

# 2.10.3 创建用水、用电、燃气哑变量
# 删除已存在的相关哑变量列
utility_columns_to_drop = [col for col in rent_price_df.columns if 
                          col.startswith('water_') or 
                          col.startswith('electricity_') or 
                          col.startswith('gas_')]
rent_price_df = rent_price_df.drop(columns=utility_columns_to_drop)

# 初始化哑变量为NaN
rent_price_df['water_civil'] = np.nan
rent_price_df['water_commercial'] = np.nan
rent_price_df['electricity_civil'] = np.nan
rent_price_df['electricity_commercial'] = np.nan
rent_price_df['gas_yes'] = np.nan

# 处理用水哑变量
def process_water_data(row):
    """处理用水数据，返回(民水, 商水)的元组"""
    # 首先检查第一标准
    if pd.notna(row['用水']):
        water_str = str(row['用水'])
        if water_str == '民水':
            return (1, 0)
        elif water_str == '商水':
            return (0, 1)
        elif '民水' in water_str and '商水' in water_str:
            return (1, 1)
    
    # 如果第一标准没有信息，检查第二标准
    if pd.notna(row['供水']):
        water_supply_str = str(row['供水'])
        if water_supply_str == '民水':
            return (1, 0)
        elif water_supply_str == '商水':
            return (0, 1)
        elif '民水' in water_supply_str and '商水' in water_supply_str:
            return (1, 1)
        elif '民水' in water_supply_str:
            return (1, 0)
        elif '商水' in water_supply_str:
            return (0, 1)
    
    # 两个标准都没有信息，返回(None, None)表示缺失
    return (None, None)

# 处理用电哑变量
def process_electricity_data(row):
    """处理用电数据，返回(民电, 商电)的元组"""
    # 首先检查第一标准
    if pd.notna(row['用电']):
        electricity_str = str(row['用电'])
        if electricity_str == '民电':
            return (1, 0)
        elif electricity_str == '商电':
            return (0, 1)
        elif '民电' in electricity_str and '商电' in electricity_str:
            return (1, 1)
    
    # 如果第一标准没有信息，检查第二标准
    if pd.notna(row['供电']):
        electricity_supply_str = str(row['供电'])
        if electricity_supply_str == '民电':
            return (1, 0)
        elif electricity_supply_str == '商电':
            return (0, 1)
        elif '民电' in electricity_supply_str and '商电' in electricity_supply_str:
            return (1, 1)
        elif '民电' in electricity_supply_str:
            return (1, 0)
        elif '商电' in electricity_supply_str:
            return (0, 1)
    
    # 两个标准都没有信息，返回(None, None)表示缺失
    return (None, None)

# 处理燃气哑变量
def process_gas_data(row):
    """处理燃气数据，返回是否有燃气(1/0)"""
    # 首先检查第一标准
    if pd.notna(row['燃气']):
        gas_str = str(row['燃气'])
        if gas_str == '有':
            return 1
        elif gas_str == '无':
            return 0
    
    # 如果第一标准没有信息，检查第二标准
    if pd.notna(row['燃气费']):
        gas_fee_str = str(row['燃气费'])
        # 检查是否包含数字（有数字表示有燃气）
        if any(char.isdigit() for char in gas_fee_str):
            return 1
    
    # 两个标准都没有信息，返回None表示缺失
    return None

# 应用处理函数
for idx, row in rent_price_df.iterrows():
    # 处理用水
    water_result = process_water_data(row)
    if water_result[0] is not None:
        rent_price_df.at[idx, 'water_civil'] = water_result[0]
        rent_price_df.at[idx, 'water_commercial'] = water_result[1]
    
    # 处理用电
    electricity_result = process_electricity_data(row)
    if electricity_result[0] is not None:
        rent_price_df.at[idx, 'electricity_civil'] = electricity_result[0]
        rent_price_df.at[idx, 'electricity_commercial'] = electricity_result[1]
    
    # 处理燃气
    gas_result = process_gas_data(row)
    if gas_result is not None:
        rent_price_df.at[idx, 'gas_yes'] = gas_result

print(f"\n已创建的用水哑变量:")
print(f"water_civil: 1表示民水，0表示非民水，NaN表示缺失")
print(f"water_commercial: 1表示商水，0表示非商水，NaN表示缺失")

print("\n用水哑变量分布:")
print(f"民水 (water_civil=1): {(rent_price_df['water_civil'] == 1).sum()}")
print(f"非民水 (water_civil=0): {(rent_price_df['water_civil'] == 0).sum()}")
print(f"缺失值: {rent_price_df['water_civil'].isna().sum()}")

print(f"\n商水 (water_commercial=1): {(rent_price_df['water_commercial'] == 1).sum()}")
print(f"非商水 (water_commercial=0): {(rent_price_df['water_commercial'] == 0).sum()}")
print(f"缺失值: {rent_price_df['water_commercial'].isna().sum()}")

# 检查同时有民水和商水的情况
both_water_mask = (rent_price_df['water_civil'] == 1) & (rent_price_df['water_commercial'] == 1)
print(f"同时有民水和商水: {both_water_mask.sum()}")

print(f"\n已创建的用电哑变量:")
print(f"electricity_civil: 1表示民电，0表示非民电，NaN表示缺失")
print(f"electricity_commercial: 1表示商电，0表示非商电，NaN表示缺失")

print("\n用电哑变量分布:")
print(f"民电 (electricity_civil=1): {(rent_price_df['electricity_civil'] == 1).sum()}")
print(f"非民电 (electricity_civil=0): {(rent_price_df['electricity_civil'] == 0).sum()}")
print(f"缺失值: {rent_price_df['electricity_civil'].isna().sum()}")

print(f"\n商电 (electricity_commercial=1): {(rent_price_df['electricity_commercial'] == 1).sum()}")
print(f"非商电 (electricity_commercial=0): {(rent_price_df['electricity_commercial'] == 0).sum()}")
print(f"缺失值: {rent_price_df['electricity_commercial'].isna().sum()}")

# 检查同时有民电和商电的情况
both_electricity_mask = (rent_price_df['electricity_civil'] == 1) & (rent_price_df['electricity_commercial'] == 1)
print(f"同时有民电和商电: {both_electricity_mask.sum()}")

print(f"\n已创建的燃气哑变量:")
print(f"gas_yes: 1表示有燃气，0表示无燃气，NaN表示缺失")

print("\n燃气哑变量分布:")
print(f"有燃气 (gas_yes=1): {(rent_price_df['gas_yes'] == 1).sum()}")
print(f"无燃气 (gas_yes=0): {(rent_price_df['gas_yes'] == 0).sum()}")
print(f"缺失值: {rent_price_df['gas_yes'].isna().sum()}")

print("\n用水用电燃气哑变量创建完成（保留缺失值）")

用水分布:
用水
民水    6794
商水     706
Name: count, dtype: int64

用电分布:
用电
民电    6806
商电     740
Name: count, dtype: int64

燃气分布:
燃气
有    8381
无     801
Name: count, dtype: int64

供水分布:
供水
民水       4313
商水/民水    2410
商水        288
Name: count, dtype: int64

供电分布:
供电
民电       4181
商电/民电    2549
商电        283
Name: count, dtype: int64

燃气费分布:
燃气费
2.61元/m³         1285
3元/m³             924
3.45元/m³          861
3.5元/m³           647
1.98元/m³          305
2.95元/m³          220
2.46元/m³          188
2.61-2.63元/m³     165
1.96元/m³          145
3-3.45元/m³        141
Name: count, dtype: int64

用水缺失值数量: 2273
用电缺失值数量: 2227
燃气缺失值数量: 591
供水缺失值数量: 2762
供电缺失值数量: 2760
燃气费缺失值数量: 3142

已创建的用水哑变量:
water_civil: 1表示民水，0表示非民水，NaN表示缺失
water_commercial: 1表示商水，0表示非商水，NaN表示缺失

用水哑变量分布:
民水 (water_civil=1): 7578
非民水 (water_civil=0): 800
缺失值: 1395

商水 (water_commercial=1): 1052
非商水 (water_commercial=0): 7326
缺失值: 1395
同时有民水和商水: 252

已创建的用电哑变量:
electricity_civil: 1表示民电，0表示非民电，NaN表示缺失
electricity_commercial: 1表示商电，0表示非商电，

In [13]:
# 2.11 采暖
# 2.11.1 分布检查
print("采暖分布:")
print(rent_price_df['采暖'].value_counts())

print("\n供暖分布:")
print(rent_price_df['供暖'].value_counts().head(10))  # 只显示前10个

# 2.11.2 缺失值检查
print(f"\n采暖缺失值数量: {rent_price_df['采暖'].isna().sum()}")
print(f"供暖缺失值数量: {rent_price_df['供暖'].isna().sum()}")

# 2.11.3 创建采暖哑变量
# 删除已存在的采暖哑变量列
heating_columns_to_drop = [col for col in rent_price_df.columns if col.startswith('heating_')]
rent_price_df = rent_price_df.drop(columns=heating_columns_to_drop)

# 初始化哑变量为NaN
rent_price_df['heating_self'] = np.nan

# 处理采暖哑变量
def process_heating_data(row):
    """处理采暖数据，返回采暖类型(0=集中供暖, 1=自采暖)"""
    # 首先检查第一标准
    if pd.notna(row['采暖']):
        heating_str = str(row['采暖'])
        if heating_str == '集中供暖':
            return 0
        elif heating_str == '自采暖':
            return 1
        elif '集中供暖' in heating_str and '自采暖' in heating_str:
            # 如果两者都有，我们将其视为自采暖，因为自采暖需要额外标记
            return 1
    
    # 如果第一标准没有信息，检查第二标准
    if pd.notna(row['供暖']):
        heating_supply_str = str(row['供暖'])
        if heating_supply_str == '集中供暖':
            return 0
        elif heating_supply_str == '自采暖':
            return 1
        elif '集中供暖' in heating_supply_str and '自采暖' in heating_supply_str:
            # 如果两者都有，我们将其视为自采暖
            return 1
        elif '集中供暖' in heating_supply_str:
            return 0
        elif '自采暖' in heating_supply_str:
            return 1
    
    # 两个标准都没有信息，返回None表示缺失
    return None

# 应用处理函数
for idx, row in rent_price_df.iterrows():
    heating_result = process_heating_data(row)
    if heating_result is not None:
        rent_price_df.at[idx, 'heating_self'] = heating_result

print(f"\n已创建的采暖哑变量:")
print(f"heating_self: 1表示自采暖，0表示集中供暖，NaN表示缺失")

print("\n采暖哑变量分布:")
print(f"集中供暖 (heating_self=0): {(rent_price_df['heating_self'] == 0).sum()}")
print(f"自采暖 (heating_self=1): {(rent_price_df['heating_self'] == 1).sum()}")
print(f"缺失值: {rent_price_df['heating_self'].isna().sum()}")

print("采暖哑变量创建完成（保留缺失值）")

采暖分布:
采暖
集中供暖    2144
自采暖     1069
Name: count, dtype: int64

供暖分布:
供暖
集中供暖            1706
自采暖             1329
集中供暖/自采暖         381
无供暖              132
集中供暖/自采暖/无供暖      16
自采暖/无供暖           14
Name: count, dtype: int64

采暖缺失值数量: 6560
供暖缺失值数量: 6195

已创建的采暖哑变量:
heating_self: 1表示自采暖，0表示集中供暖，NaN表示缺失

采暖哑变量分布:
集中供暖 (heating_self=0): 2212
自采暖 (heating_self=1): 1587
缺失值: 5974
采暖哑变量创建完成（保留缺失值）


In [14]:
# 2.12 租期变量
# 2.12.1 分布检查
print("租期分布:")
print(rent_price_df['租期'].value_counts())  # 只显示前10个最常见的值
# 在初步调研时需要显示所有行，可以运行以下代码
# import pandas as pd
# pd.set_option('display.max_rows', None)
# print(rent_price_df['租期'].value_counts())
# pd.reset_option('display.max_rows')

# 2.12.2 缺失值检查
print(f"\n租期缺失值数量: {rent_price_df['租期'].isna().sum()}")

# 2.12.3 创建租期变量
# 删除已存在的租期变量列
lease_columns_to_drop = [col for col in rent_price_df.columns if 
                        col in ['lease_min_months', 'lease_max_months', 'lease_avg_months']]
rent_price_df = rent_price_df.drop(columns=lease_columns_to_drop)

# 初始化租期变量为NaN
rent_price_df['lease_min_months'] = np.nan
rent_price_df['lease_max_months'] = np.nan
rent_price_df['lease_avg_months'] = np.nan

def convert_lease_period(lease_str):
    """
    将租期字符串转换为最小月数、最大月数和平均月数
    处理逻辑:
    - "X年以内": 最小=1月, 最大=X年, 平均=(1+X年)/2
    - "X年以上": 最小=X年, 最大=60月(5年), 平均=(X年+60)/2
    - "X~Y年": 最小=X年, 最大=Y年, 平均=(X+Y)/2年
    - "X个月": 最小=最大=平均=X月
    - "X~Y个月": 最小=X月, 最大=Y月, 平均=(X+Y)/2月
    - 其他格式类似处理
    """
    if pd.isna(lease_str):
        return (np.nan, np.nan, np.nan)
    
    lease_str = str(lease_str).strip()
    
    # 初始化默认值
    min_months = np.nan
    max_months = np.nan
    avg_months = np.nan
    
    try:
        # 处理"X年以内"格式
        if '年以内' in lease_str:
            years = float(lease_str.replace('年以内', ''))
            min_months = 1  # 最小1个月
            max_months = years * 12
            avg_months = (min_months + max_months) / 2
        
        # 处理"X年以上"格式
        elif '年以上' in lease_str:
            years = float(lease_str.replace('年以上', ''))
            min_months = years * 12
            max_months = 60  # 最大5年(60个月)
            avg_months = (min_months + max_months) / 2
        
        # 处理"X个月以上"格式
        elif '个月以上' in lease_str:
            months = float(lease_str.replace('个月以上', ''))
            min_months = months
            max_months = 60  # 最大5年(60个月)
            avg_months = (min_months + max_months) / 2
        
        # 处理"X个月以内"格式
        elif '个月以内' in lease_str:
            months = float(lease_str.replace('个月以内', ''))
            min_months = 1  # 最小1个月
            max_months = months
            avg_months = (min_months + max_months) / 2
        
        # 处理"X~Y年"格式
        elif '~' in lease_str and '年' in lease_str:
            parts = lease_str.replace('年', '').split('~')
            min_years = float(parts[0])
            max_years = float(parts[1])
            min_months = min_years * 12
            max_months = max_years * 12
            avg_months = (min_months + max_months) / 2
        
        # 处理"X~Y个月"格式
        elif '~' in lease_str and '个月' in lease_str:
            parts = lease_str.replace('个月', '').split('~')
            min_months = float(parts[0])
            max_months = float(parts[1])
            avg_months = (min_months + max_months) / 2
        
        # 处理单个年份"X年"格式
        elif '年' in lease_str and '~' not in lease_str and '以内' not in lease_str and '以上' not in lease_str:
            years = float(lease_str.replace('年', ''))
            min_months = years * 12
            max_months = years * 12
            avg_months = years * 12
        
        # 处理单个月份"X个月"格式
        elif '个月' in lease_str and '~' not in lease_str and '以内' not in lease_str and '以上' not in lease_str:
            months = float(lease_str.replace('个月', ''))
            min_months = months
            max_months = months
            avg_months = months
        
        # 处理纯数字(假设为月)
        elif lease_str.replace('.', '').isdigit():
            months = float(lease_str)
            min_months = months
            max_months = months
            avg_months = months
        
        # 处理"X个月以内"但缺少"以内"的情况(如"1年")
        elif '年' in lease_str and len(lease_str) < 5:
            years = float(lease_str.replace('年', ''))
            min_months = years * 12
            max_months = years * 12
            avg_months = years * 12
            
    except (ValueError, IndexError) as e:
        # 如果解析失败，保持为NaN
        print(f"警告: 无法解析租期 '{lease_str}': {e}")
        min_months, max_months, avg_months = np.nan, np.nan, np.nan
    
    return (min_months, max_months, avg_months)

# 只对非空租期进行处理
valid_mask = rent_price_df['租期'].notna()

# 应用转换函数
for idx in rent_price_df[valid_mask].index:
    lease_str = rent_price_df.loc[idx, '租期']
    min_months, max_months, avg_months = convert_lease_period(lease_str)
    
    rent_price_df.loc[idx, 'lease_min_months'] = min_months
    rent_price_df.loc[idx, 'lease_max_months'] = max_months
    rent_price_df.loc[idx, 'lease_avg_months'] = avg_months

print(f"\n已创建的租期变量:")
print(f"lease_min_months: 最小租期(月)，NaN表示缺失")
print(f"lease_max_months: 最大租期(月)，NaN表示缺失")
print(f"lease_avg_months: 平均租期(月)，NaN表示缺失")

print("\n租期变量统计:")
lease_features = ['lease_min_months', 'lease_max_months', 'lease_avg_months']
for feature in lease_features:
    print(f"{feature}:")
    print(f"  非缺失值数量: {rent_price_df[feature].notna().sum()}")
    if rent_price_df[feature].notna().sum() > 0:
        print(f"  平均值: {rent_price_df[feature].mean():.2f}")
        print(f"  标准差: {rent_price_df[feature].std():.2f}")
        print(f"  最小值: {rent_price_df[feature].min()}")
        print(f"  最大值: {rent_price_df[feature].max()}")
    print(f"  缺失值数量: {rent_price_df[feature].isna().sum()}")

print("租期变量创建完成（保留缺失值）")

# 显示一些转换示例
print("\n租期转换示例:")
sample_data = rent_price_df[valid_mask][['租期', 'lease_min_months', 'lease_max_months', 'lease_avg_months']].head(10)
print(sample_data)

租期分布:
租期
1年以内       2271
2年以内        495
1~2年        482
1年          434
3年以内        390
3年以上        100
1~12个月       68
6~12个月       57
7~12个月       45
3~12个月       38
1~3年         24
2~12个月       24
1~36个月       23
1年以上         17
4~7个月        13
1~24个月       12
1~3个月        11
6~24个月       10
4~12个月       10
3个月以上         7
3~24个月        6
3~36个月        6
2~4个月         6
1个月以上         5
6个月以上         4
2年            4
8~12个月        4
11~12个月       3
2~24个月        3
8~36个月        3
1个月以内         3
1~7个月         2
6~36个月        2
10~12个月       2
6~8个月         1
4个月           1
6个月           1
6~7个月         1
3~6个月         1
2~7个月         1
9~10个月        1
5~36个月        1
3个月           1
1个月           1
4~6个月         1
9~12个月        1
3年            1
5~6个月         1
Name: count, dtype: int64

租期缺失值数量: 5175

已创建的租期变量:
lease_min_months: 最小租期(月)，NaN表示缺失
lease_max_months: 最大租期(月)，NaN表示缺失
lease_avg_months: 平均租期(月)，NaN表示缺失

租期变量统计:
lease_min_months:
  非缺失值数量: 4598
  平均值: 4.30
  标准差: 6.54
  最

In [15]:
# 2.13 配套设施
# 2.13.1 分布检查
print("配套设施分布:")
print(rent_price_df['配套设施'].value_counts())  # 只显示前10个最常见的值
# 在初步调研时需要显示所有行，可以运行以下代码
# import pandas as pd
# pd.set_option('display.max_rows', None)
# print(rent_price_df['配套设施'].value_counts())
# pd.reset_option('display.max_rows')

# 2.13.2 缺失值检查
print(f"\n配套设施缺失值数量: {rent_price_df['配套设施'].isna().sum()}")

# 2.13.3 创建配套设施哑变量
# 定义所有可能的设施名称
facility_list = ['洗衣机', '空调', '衣柜', '电视', '冰箱', '热水器', '床', '天然气', '暖气', '宽带']

# 删除已存在的配套设施哑变量列
facility_columns_to_drop = [col for col in rent_price_df.columns if col.startswith('facility_')]
rent_price_df = rent_price_df.drop(columns=facility_columns_to_drop)

# 初始化所有设施哑变量为NaN
for facility in facility_list:
    rent_price_df[f'facility_{facility}'] = np.nan

# 只对非空配套设施进行处理
valid_mask = rent_price_df['配套设施'].notna()

# 应用处理函数
for idx in rent_price_df[valid_mask].index:
    facility_str = rent_price_df.loc[idx, '配套设施']
    
    # 检查每个设施是否在字符串中
    for facility in facility_list:
        if facility in facility_str:
            rent_price_df.loc[idx, f'facility_{facility}'] = 1
        else:
            rent_price_df.loc[idx, f'facility_{facility}'] = 0

print(f"\n已创建的配套设施哑变量:")
for facility in facility_list:
    print(f"facility_{facility}: 1表示有{facility}，0表示无{facility}，NaN表示缺失")

print("\n配套设施哑变量分布:")
for facility in facility_list:
    col_name = f'facility_{facility}'
    print(f"{facility}:")
    print(f"  有 {col_name}=1: {(rent_price_df[col_name] == 1).sum()}")
    print(f"  无 {col_name}=0: {(rent_price_df[col_name] == 0).sum()}")
    print(f"  缺失值: {rent_price_df[col_name].isna().sum()}")

print("配套设施哑变量创建完成（保留缺失值）")

# 显示一些转换示例
print("\n配套设施转换示例:")
sample_data = rent_price_df[valid_mask][['配套设施'] + [f'facility_{facility}' for facility in facility_list]].head(5)
print(sample_data)

配套设施分布:
配套设施
洗衣机、空调、衣柜、电视、冰箱、热水器、床、天然气        801
洗衣机、空调、衣柜、冰箱、热水器、床               441
洗衣机、空调、衣柜、冰箱、热水器、床、天然气           392
洗衣机、空调、衣柜、电视、冰箱、热水器、床、暖气、天然气     330
洗衣机、空调、衣柜、电视、冰箱、热水器、床            317
                                ... 
空调、衣柜、冰箱、热水器、床、宽带                  1
空调、衣柜、床、宽带、天然气                     1
洗衣机、空调、衣柜、电视、冰箱、热水器、暖气、宽带、天然气      1
洗衣机、空调、衣柜、电视、热水器、宽带                1
洗衣机、空调、衣柜、冰箱、宽带                    1
Name: count, Length: 361, dtype: int64

配套设施缺失值数量: 3694

已创建的配套设施哑变量:
facility_洗衣机: 1表示有洗衣机，0表示无洗衣机，NaN表示缺失
facility_空调: 1表示有空调，0表示无空调，NaN表示缺失
facility_衣柜: 1表示有衣柜，0表示无衣柜，NaN表示缺失
facility_电视: 1表示有电视，0表示无电视，NaN表示缺失
facility_冰箱: 1表示有冰箱，0表示无冰箱，NaN表示缺失
facility_热水器: 1表示有热水器，0表示无热水器，NaN表示缺失
facility_床: 1表示有床，0表示无床，NaN表示缺失
facility_天然气: 1表示有天然气，0表示无天然气，NaN表示缺失
facility_暖气: 1表示有暖气，0表示无暖气，NaN表示缺失
facility_宽带: 1表示有宽带，0表示无宽带，NaN表示缺失

配套设施哑变量分布:
洗衣机:
  有 facility_洗衣机=1: 5080
  无 facility_洗衣机=0: 999
  缺失值: 3694
空调:
  有 facility_空调=1: 5159
  无 facility_空调=0: 920
  缺失值: 3694
衣柜:
  有 facil

In [16]:
# 2.15 建筑年代
# 2.15.1 分布检查
print("建筑年代分布:")
print(rent_price_df['建筑年代'].value_counts()) 
# 在初步调研时需要显示所有行，可以运行以下代码
# import pandas as pd
# pd.set_option('display.max_rows', None)
# print(rent_price_df['建筑年代'].value_counts())
# pd.reset_option('display.max_rows')

# 2.15.2 缺失值检查
print(f"\n建筑年代缺失值数量: {rent_price_df['建筑年代'].isna().sum()}")

# 2.15.3 创建房龄变量
# 删除已存在的房龄变量列
age_columns_to_drop = [col for col in rent_price_df.columns if col in ['building_age']]
rent_price_df = rent_price_df.drop(columns=age_columns_to_drop)

# 初始化房龄变量为NaN
rent_price_df['building_age'] = np.nan

def extract_building_age(building_year_str):
    """
    从建筑年代字符串中提取建筑结束年份并计算房龄
    处理逻辑:
    - 单个年份: "2008年" → 结束年份=2008
    - 精确区间: "2008-2014年" → 结束年份=2014
    - 跨度区间: "1980-1999年" → 结束年份=1999
    - 房龄 = 2025 - 结束年份 (假设数据收集年份为2025年)
    """
    if pd.isna(building_year_str):
        return np.nan
    
    building_year_str = str(building_year_str).strip()
    
    try:
        # 假设数据收集年份为2025年
        current_year = 2025
        
        # 处理单个年份格式
        if '年' in building_year_str and '-' not in building_year_str:
            year = int(building_year_str.replace('年', ''))
            return current_year - year
        
        # 处理区间格式
        elif '-' in building_year_str and '年' in building_year_str:
            # 提取年份范围
            year_range = building_year_str.replace('年', '').split('-')
            
            # 确保有两个年份
            if len(year_range) == 2:
                start_year = int(year_range[0])
                end_year = int(year_range[1])
                
                # 使用结束年份计算房龄
                return current_year - end_year
        
        # 其他格式无法解析，返回NaN
        return np.nan
        
    except (ValueError, IndexError) as e:
        # 如果解析失败，返回NaN
        print(f"警告: 无法解析建筑年代 '{building_year_str}': {e}")
        return np.nan

# 只对非空建筑年代进行处理
valid_mask = rent_price_df['建筑年代'].notna()

# 应用转换函数
for idx in rent_price_df[valid_mask].index:
    building_year_str = rent_price_df.loc[idx, '建筑年代']
    building_age = extract_building_age(building_year_str)
    rent_price_df.loc[idx, 'building_age'] = building_age

print(f"\n已创建的房龄变量:")
print(f"building_age: 房龄(年)，基于建筑结束年份计算，NaN表示缺失")

print("\n房龄变量统计:")
print(f"非缺失值数量: {rent_price_df['building_age'].notna().sum()}")
if rent_price_df['building_age'].notna().sum() > 0:
    print(f"平均值: {rent_price_df['building_age'].mean():.2f}")
    print(f"标准差: {rent_price_df['building_age'].std():.2f}")
    print(f"最小值: {rent_price_df['building_age'].min()}")
    print(f"最大值: {rent_price_df['building_age'].max()}")
    print(f"分布:")
    print(rent_price_df['building_age'].value_counts().sort_index().head(20))  # 只显示前20个最常见的房龄
print(f"缺失值数量: {rent_price_df['building_age'].isna().sum()}")

print("房龄变量创建完成（保留缺失值）")

# 显示一些转换示例
print("\n建筑年代转换示例:")
sample_data = rent_price_df[valid_mask][['建筑年代', 'building_age']].head(10)
print(sample_data)

建筑年代分布:
建筑年代
2019年         99
2015-2018年    84
2008年         81
2000-2012年    77
2008-2014年    74
              ..
2005-2013年     1
1999-2001年     1
1997-2004年     1
1992-1999年     1
1977-2002年     1
Name: count, Length: 594, dtype: int64

建筑年代缺失值数量: 3381

已创建的房龄变量:
building_age: 房龄(年)，基于建筑结束年份计算，NaN表示缺失

房龄变量统计:
非缺失值数量: 6392
平均值: 15.93
标准差: 8.63
最小值: 3.0
最大值: 55.0
分布:
building_age
3.0      46
4.0      59
5.0     251
6.0     423
7.0     403
8.0     325
9.0     446
10.0    335
11.0    241
12.0    247
13.0    276
14.0    154
15.0    192
16.0    277
17.0    294
18.0    164
19.0    178
20.0    175
21.0    232
22.0    215
Name: count, dtype: int64
缺失值数量: 3381
房龄变量创建完成（保留缺失值）

建筑年代转换示例:
          建筑年代  building_age
0   2011-2019年           6.0
1   1985-2008年          17.0
3   2004-2009年          16.0
5   2010-2015年          10.0
7   1983-1990年          35.0
9   2012-2017年           8.0
12  2012-2018年           7.0
13  2010-2015年          10.0
14       2019年           6.0
15       2008年      

In [17]:
# 2.16 小区规模（房屋总数和栋数）
# 2.16.1 分布检查
print("房屋总数分布:")
print(rent_price_df['房屋总数'].value_counts().head(20))  # 只显示前20个最常见的值

print("\n楼栋总数分布:")
print(rent_price_df['楼栋总数'].value_counts().head(20))  # 只显示前20个最常见的值

# 2.16.2 缺失值检查
print(f"\n房屋总数缺失值数量: {rent_price_df['房屋总数'].isna().sum()}")
print(f"楼栋总数缺失值数量: {rent_price_df['楼栋总数'].isna().sum()}")

# 2.16.3 创建小区规模变量
# 删除已存在的小区规模变量列
community_columns_to_drop = [col for col in rent_price_df.columns if 
                            col in ['household_total', 'building_total']]
rent_price_df = rent_price_df.drop(columns=community_columns_to_drop)

# 初始化小区规模变量为NaN
rent_price_df['household_total'] = np.nan
rent_price_df['building_total'] = np.nan

def extract_community_scale(household_str, building_str):
    """
    从房屋总数和楼栋总数字符串中提取数值
    处理逻辑:
    - 提取数字部分，去除"户"、"栋"等单位
    - 转换为数值类型
    """
    household_value = np.nan
    building_value = np.nan
    
    # 处理房屋总数
    if pd.notna(household_str):
        household_str = str(household_str).strip()
        try:
            # 提取数字部分
            household_match = re.search(r'(\d+)', household_str)
            if household_match:
                household_value = int(household_match.group(1))
        except (ValueError, AttributeError) as e:
            print(f"警告: 无法解析房屋总数 '{household_str}': {e}")
    
    # 处理楼栋总数
    if pd.notna(building_str):
        building_str = str(building_str).strip()
        try:
            # 提取数字部分
            building_match = re.search(r'(\d+)', building_str)
            if building_match:
                building_value = int(building_match.group(1))
        except (ValueError, AttributeError) as e:
            print(f"警告: 无法解析楼栋总数 '{building_str}': {e}")
    
    return household_value, building_value

# 只对非空数据进行处理
valid_household_mask = rent_price_df['房屋总数'].notna()
valid_building_mask = rent_price_df['楼栋总数'].notna()

# 应用转换函数
for idx in rent_price_df.index:
    household_str = rent_price_df.loc[idx, '房屋总数']
    building_str = rent_price_df.loc[idx, '楼栋总数']
    
    household_value, building_value = extract_community_scale(household_str, building_str)
    
    if pd.notna(household_value):
        rent_price_df.loc[idx, 'household_total'] = household_value
    
    if pd.notna(building_value):
        rent_price_df.loc[idx, 'building_total'] = building_value

print(f"\n已创建的小区规模变量:")
print(f"household_total: 房屋总户数，NaN表示缺失")
print(f"building_total: 楼栋总数，NaN表示缺失")

print("\n小区规模变量统计:")
print("household_total:")
print(f"  非缺失值数量: {rent_price_df['household_total'].notna().sum()}")
if rent_price_df['household_total'].notna().sum() > 0:
    print(f"  平均值: {rent_price_df['household_total'].mean():.2f}")
    print(f"  标准差: {rent_price_df['household_total'].std():.2f}")
    print(f"  最小值: {rent_price_df['household_total'].min()}")
    print(f"  最大值: {rent_price_df['household_total'].max()}")
print(f"  缺失值数量: {rent_price_df['household_total'].isna().sum()}")

print("\nbuilding_total:")
print(f"  非缺失值数量: {rent_price_df['building_total'].notna().sum()}")
if rent_price_df['building_total'].notna().sum() > 0:
    print(f"  平均值: {rent_price_df['building_total'].mean():.2f}")
    print(f"  标准差: {rent_price_df['building_total'].std():.2f}")
    print(f"  最小值: {rent_price_df['building_total'].min()}")
    print(f"  最大值: {rent_price_df['building_total'].max()}")
print(f"  缺失值数量: {rent_price_df['building_total'].isna().sum()}")

print("小区规模变量创建完成（保留缺失值）")

# 显示一些转换示例
print("\n小区规模转换示例:")
sample_data = rent_price_df[['房屋总数', '楼栋总数', 'household_total', 'building_total']].head(10)
print(sample_data)

房屋总数分布:
房屋总数
3983户    94
1761户    73
2928户    54
4639户    54
6458户    52
2652户    46
1445户    43
5184户    41
1804户    41
2925户    40
9342户    40
1870户    38
3139户    38
7387户    37
2117户    37
2283户    37
1160户    33
5499户    33
2154户    32
2587户    31
Name: count, dtype: int64

楼栋总数分布:
楼栋总数
3栋     488
2栋     452
8栋     407
6栋     380
10栋    356
7栋     355
1栋     332
4栋     330
5栋     323
13栋    316
12栋    309
9栋     277
15栋    219
11栋    209
17栋    204
19栋    197
16栋    195
23栋    162
22栋    162
14栋    160
Name: count, dtype: int64

房屋总数缺失值数量: 923
楼栋总数缺失值数量: 923

已创建的小区规模变量:
household_total: 房屋总户数，NaN表示缺失
building_total: 楼栋总数，NaN表示缺失

小区规模变量统计:
household_total:
  非缺失值数量: 8850
  平均值: 1882.72
  标准差: 1745.16
  最小值: 1.0
  最大值: 12669.0
  缺失值数量: 923

building_total:
  非缺失值数量: 8850
  平均值: 28.49
  标准差: 54.58
  最小值: 1.0
  最大值: 734.0
  缺失值数量: 923
小区规模变量创建完成（保留缺失值）

小区规模转换示例:
    房屋总数 楼栋总数  household_total  building_total
0   992户  12栋            992.0            12.0
1   806户  15栋            80

In [18]:
# 2.17 绿化率和容积率
# 2.17.1 分布检查
print("绿化率分布:")
print(rent_price_df['绿化率'].value_counts().head(10))  # 只显示前10个最常见的值

print("\n容积率分布:")
print(rent_price_df['容积率'].value_counts().head(10))  # 只显示前10个最常见的值

# 2.17.2 缺失值检查
print(f"\n绿化率缺失值数量: {rent_price_df['绿化率'].isna().sum()}")
print(f"容积率缺失值数量: {rent_price_df['容积率'].isna().sum()}")

# 2.17.3 创建绿化率和容积率变量
# 删除已存在的绿化率和容积率变量列
greening_columns_to_drop = [col for col in rent_price_df.columns if 
                           col in ['greening_rate', 'plot_ratio']]
rent_price_df = rent_price_df.drop(columns=greening_columns_to_drop)

# 创建绿化率和容积率变量
# 保留缺失值为NaN，不进行填充

# 处理绿化率 - 移除百分号并转换为浮点数
rent_price_df['greening_rate'] = np.nan
valid_greening_mask = rent_price_df['绿化率'].notna()
rent_price_df.loc[valid_greening_mask, 'greening_rate'] = (
    rent_price_df.loc[valid_greening_mask, '绿化率']
    .astype(str)
    .str.replace('%', '')
    .astype(float) / 100
)

# 处理容积率 - 直接转换为浮点数
rent_price_df['plot_ratio'] = np.nan
valid_plot_mask = rent_price_df['容积率'].notna()
rent_price_df.loc[valid_plot_mask, 'plot_ratio'] = (
    rent_price_df.loc[valid_plot_mask, '容积率'].astype(float)
)

print(f"\n已创建的变量:")
print(f"greening_rate: 绿化率（小数形式），NaN表示缺失")
print(f"plot_ratio: 容积率，NaN表示缺失")

print("\n绿化率和容积率变量统计:")
print("greening_rate:")
print(f"  非缺失值数量: {rent_price_df['greening_rate'].notna().sum()}")
if rent_price_df['greening_rate'].notna().sum() > 0:
    print(f"  平均值: {rent_price_df['greening_rate'].mean():.3f}")
    print(f"  标准差: {rent_price_df['greening_rate'].std():.3f}")
    print(f"  最小值: {rent_price_df['greening_rate'].min():.3f}")
    print(f"  最大值: {rent_price_df['greening_rate'].max():.3f}")
print(f"  缺失值数量: {rent_price_df['greening_rate'].isna().sum()}")

print("\nplot_ratio:")
print(f"  非缺失值数量: {rent_price_df['plot_ratio'].notna().sum()}")
if rent_price_df['plot_ratio'].notna().sum() > 0:
    print(f"  平均值: {rent_price_df['plot_ratio'].mean():.3f}")
    print(f"  标准差: {rent_price_df['plot_ratio'].std():.3f}")
    print(f"  最小值: {rent_price_df['plot_ratio'].min():.3f}")
    print(f"  最大值: {rent_price_df['plot_ratio'].max():.3f}")
print(f"  缺失值数量: {rent_price_df['plot_ratio'].isna().sum()}")

print("绿化率和容积率变量创建完成（保留缺失值）")

绿化率分布:
绿化率
0.30    1813
0.35    1482
0.40     479
0.20     285
0.45     276
0.25     272
0.36     198
0.10     168
0.50      96
0.38      95
Name: count, dtype: int64

容积率分布:
容积率
2.0    572
3.0    520
2.5    469
1.5    251
1.0    175
3.5    175
3.2    173
4.0    160
2.2    158
1.8    156
Name: count, dtype: int64

绿化率缺失值数量: 3019
容积率缺失值数量: 2988

已创建的变量:
greening_rate: 绿化率（小数形式），NaN表示缺失
plot_ratio: 容积率，NaN表示缺失

绿化率和容积率变量统计:
greening_rate:
  非缺失值数量: 6754
  平均值: 0.004
  标准差: 0.028
  最小值: 0.000
  最大值: 1.050
  缺失值数量: 3019

plot_ratio:
  非缺失值数量: 6785
  平均值: 2.945
  标准差: 1.671
  最小值: 0.020
  最大值: 18.800
  缺失值数量: 2988
绿化率和容积率变量创建完成（保留缺失值）


In [19]:
# 2.18 物业费处理
# 2.18.1 分布检查
print("物业费分布:")
print(rent_price_df['物业费'].value_counts())
print(f"\n物业费缺失值数量: {rent_price_df['物业费'].isna().sum()}")

# 2.18.2 创建物业费变量
# 删除已存在的物业费变量列
property_columns_to_drop = [col for col in rent_price_df.columns if 
                           col in ['property_fee_avg']]
rent_price_df = rent_price_df.drop(columns=property_columns_to_drop)

# 初始化物业费变量为NaN
rent_price_df['property_fee_avg'] = np.nan

def extract_property_fee(property_fee_str):
    """
    从物业费字符串中提取平均值
    处理逻辑:
    - 单个数值: "2.8元/月/㎡" → 平均值=2.8
    - 区间数值: "2.38-2.98元/月/㎡" → 平均值=(2.38+2.98)/2=2.68
    - 特殊格式: "0.8-1元/月/㎡" → 平均值=(0.8+1)/2=0.9
    """
    if pd.isna(property_fee_str):
        return np.nan
    
    property_fee_str = str(property_fee_str).strip()
    
    try:
        # 方法1: 直接匹配数字和区间格式
        # 匹配单个数字: "2.8元/月/㎡"
        single_match = re.match(r'^(\d+\.?\d*)元/月/㎡$', property_fee_str)
        if single_match:
            return float(single_match.group(1))
        
        # 匹配区间格式: "2.38-2.98元/月/㎡"
        range_match = re.match(r'^(\d+\.?\d*)-(\d+\.?\d*)元/月/㎡$', property_fee_str)
        if range_match:
            fee_min = float(range_match.group(1))
            fee_max = float(range_match.group(2))
            return (fee_min + fee_max) / 2
        
        # 方法2: 提取所有数字并处理
        numbers = re.findall(r'\d+\.?\d*', property_fee_str)
        if len(numbers) == 1:
            # 只有一个数字
            return float(numbers[0])
        elif len(numbers) == 2:
            # 有两个数字，认为是区间
            fee_min = float(numbers[0])
            fee_max = float(numbers[1])
            return (fee_min + fee_max) / 2
        else:
            # 无法解析的格式
            return np.nan
            
    except (ValueError, IndexError, AttributeError) as e:
        # 如果解析失败，返回NaN
        print(f"警告: 无法解析物业费 '{property_fee_str}': {e}")
        return np.nan

# 应用转换函数
for idx in rent_price_df.index:
    property_fee_str = rent_price_df.loc[idx, '物业费']
    if pd.notna(property_fee_str):
        property_fee_avg = extract_property_fee(property_fee_str)
        rent_price_df.loc[idx, 'property_fee_avg'] = property_fee_avg

print(f"\n物业费变量统计:")
print(f"非缺失值数量: {rent_price_df['property_fee_avg'].notna().sum()}")
if rent_price_df['property_fee_avg'].notna().sum() > 0:
    print(f"平均值: {rent_price_df['property_fee_avg'].mean():.2f}")
    print(f"标准差: {rent_price_df['property_fee_avg'].std():.2f}")
    print(f"最小值: {rent_price_df['property_fee_avg'].min():.2f}")
    print(f"最大值: {rent_price_df['property_fee_avg'].max():.2f}")
    print(f"中位数: {rent_price_df['property_fee_avg'].median():.2f}")
    
    # 显示分布情况
    print(f"\n物业费分布概况:")
    fee_stats = rent_price_df['property_fee_avg'].describe()
    print(fee_stats)
    
    # 按区间显示分布
    bins = [0, 1, 2, 3, 4, 5, 10, 20, 50, 100, float('inf')]
    labels = ['0-1', '1-2', '2-3', '3-4', '4-5', '5-10', '10-20', '20-50', '50-100', '100+']
    fee_ranges = pd.cut(rent_price_df['property_fee_avg'], bins=bins, labels=labels, right=False)
    print(f"\n物业费区间分布:")
    print(fee_ranges.value_counts().sort_index())

print(f"缺失值数量: {rent_price_df['property_fee_avg'].isna().sum()}")

print("物业费变量创建完成（保留缺失值）")

物业费分布:
物业费
2.8元/月/㎡          243
0.8元/月/㎡          218
1元/月/㎡            143
1.2元/月/㎡          138
1.8元/月/㎡          132
                 ... 
0.6-1.7元/月/㎡        1
0.35-1.45元/月/㎡      1
1-1.25元/月/㎡         1
11元/月/㎡             1
1.5-1.7元/月/㎡        1
Name: count, Length: 778, dtype: int64

物业费缺失值数量: 2804

物业费变量统计:
非缺失值数量: 6969
平均值: 2.71
标准差: 3.68
最小值: 0.20
最大值: 50.60
中位数: 2.05

物业费分布概况:
count    6969.000000
mean        2.707774
std         3.675572
min         0.200000
25%         1.200000
50%         2.050000
75%         3.000000
max        50.600000
Name: property_fee_avg, dtype: float64

物业费区间分布:
property_fee_avg
0-1       1112
1-2       2154
2-3       1843
3-4       1029
4-5        296
5-10       389
10-20      104
20-50       24
50-100      18
100+         0
Name: count, dtype: int64
缺失值数量: 2804
物业费变量创建完成（保留缺失值）


In [20]:
# 2.19 建筑结构
# 2.19.1 分布检查
print("建筑结构分布:")
print(rent_price_df['建筑结构'].value_counts())

# 2.19.2 缺失值检查
print(f"\n建筑结构缺失值数量: {rent_price_df['建筑结构'].isna().sum()}")

# 2.19.3 创建建筑结构哑变量
# 删除已存在的建筑结构哑变量列
structure_columns_to_drop = [col for col in rent_price_df.columns if col.startswith('structure_')]
rent_price_df = rent_price_df.drop(columns=structure_columns_to_drop)

# 创建四个主要的建筑结构哑变量
rent_price_df['structure_tower'] = 0  # 塔楼
rent_price_df['structure_slab'] = 0   # 板楼  
rent_price_df['structure_combined'] = 0  # 塔板结合
rent_price_df['structure_bungalow'] = 0  # 平房

# 根据建筑结构内容设置哑变量
for idx in rent_price_df.index:
    structure = rent_price_df.loc[idx, '建筑结构']
    if pd.notna(structure):
        structure_str = str(structure).strip()
        
        # 设置塔楼哑变量
        if '塔楼' in structure_str:
            rent_price_df.loc[idx, 'structure_tower'] = 1
            
        # 设置板楼哑变量  
        if '板楼' in structure_str:
            rent_price_df.loc[idx, 'structure_slab'] = 1
            
        # 设置塔板结合哑变量
        if '塔板结合' in structure_str:
            rent_price_df.loc[idx, 'structure_combined'] = 1
            
        # 设置平房哑变量
        if '平房' in structure_str:
            rent_price_df.loc[idx, 'structure_bungalow'] = 1

print(f"\n已创建的建筑结构哑变量:")
print(f"structure_tower: 1表示包含塔楼，0表示不包含")
print(f"structure_slab: 1表示包含板楼，0表示不包含") 
print(f"structure_combined: 1表示包含塔板结合，0表示不包含")
print(f"structure_bungalow: 1表示包含平房，0表示不包含")

print("\n建筑结构哑变量分布:")
print(f"包含塔楼 (structure_tower=1): {(rent_price_df['structure_tower'] == 1).sum()}")
print(f"包含板楼 (structure_slab=1): {(rent_price_df['structure_slab'] == 1).sum()}")
print(f"包含塔板结合 (structure_combined=1): {(rent_price_df['structure_combined'] == 1).sum()}")
print(f"包含平房 (structure_bungalow=1): {(rent_price_df['structure_bungalow'] == 1).sum()}")

print("建筑结构哑变量创建完成")

建筑结构分布:
建筑结构
板楼               1800
塔楼               1591
塔楼/板楼/塔板结合        893
塔楼/板楼             861
塔板结合              714
板楼/塔板结合           588
塔楼/塔板结合           320
塔楼/板楼/塔板结合/平房      95
塔楼/板楼/平房           66
板楼/平房              65
塔楼/平房              41
板楼/塔板结合/平房         21
塔板结合/平房            13
平房                  6
Name: count, dtype: int64

建筑结构缺失值数量: 2699

已创建的建筑结构哑变量:
structure_tower: 1表示包含塔楼，0表示不包含
structure_slab: 1表示包含板楼，0表示不包含
structure_combined: 1表示包含塔板结合，0表示不包含
structure_bungalow: 1表示包含平房，0表示不包含

建筑结构哑变量分布:
包含塔楼 (structure_tower=1): 3867
包含板楼 (structure_slab=1): 4389
包含塔板结合 (structure_combined=1): 2644
包含平房 (structure_bungalow=1): 307
建筑结构哑变量创建完成


In [21]:
# 2.20 燃气费
# 2.20.1 分布检查
print("燃气费分布:")
print(rent_price_df['燃气费'].value_counts())

# 在初步调研时需要显示所有行，可以运行以下代码
# import pandas as pd
# pd.set_option('display.max_rows', None)
# print(rent_price_df['燃气费'].value_counts())
# pd.reset_option('display.max_rows')

# 2.20.2 创建燃气费变量
# 删除已存在的燃气费变量列
gas_columns_to_drop = [col for col in rent_price_df.columns if 
                      col in ['gas_fee_avg']]
rent_price_df = rent_price_df.drop(columns=gas_columns_to_drop)

# 初始化燃气费变量为NaN
rent_price_df['gas_fee_avg'] = np.nan

def extract_gas_fee(gas_fee_str):
    """
    从燃气费字符串中提取平均值
    处理逻辑:
    - 单个数值: "2.61元/m³" → 平均值=2.61
    - 区间数值: "2.61-2.63元/m³" → 平均值=(2.61+2.63)/2=2.62
    - 单位统一为: 元/m³
    """
    if pd.isna(gas_fee_str):
        return np.nan
    
    gas_fee_str = str(gas_fee_str).strip()
    
    try:
        # 处理单个数值格式
        if '元/m³' in gas_fee_str and '-' not in gas_fee_str:
            # 提取数字部分
            fee_match = re.search(r'(\d+\.?\d*)', gas_fee_str)
            if fee_match:
                return float(fee_match.group(1))
        
        # 处理区间格式
        elif '-' in gas_fee_str and '元/m³' in gas_fee_str:
            # 提取两个数字
            fee_range = re.findall(r'(\d+\.?\d*)', gas_fee_str)
            if len(fee_range) == 2:
                fee_min = float(fee_range[0])
                fee_max = float(fee_range[1])
                # 计算平均值
                return (fee_min + fee_max) / 2
        
        # 其他格式无法解析，返回NaN
        return np.nan
        
    except (ValueError, IndexError, AttributeError) as e:
        # 如果解析失败，返回NaN
        print(f"警告: 无法解析燃气费 '{gas_fee_str}': {e}")
        return np.nan

# 应用转换函数
for idx in rent_price_df.index:
    gas_fee_str = rent_price_df.loc[idx, '燃气费']
    if pd.notna(gas_fee_str):
        gas_fee_avg = extract_gas_fee(gas_fee_str)
        rent_price_df.loc[idx, 'gas_fee_avg'] = gas_fee_avg

print(f"\n燃气费变量统计:")
print(f"非缺失值数量: {rent_price_df['gas_fee_avg'].notna().sum()}")
if rent_price_df['gas_fee_avg'].notna().sum() > 0:
    print(f"平均值: {rent_price_df['gas_fee_avg'].mean():.2f}")
    print(f"标准差: {rent_price_df['gas_fee_avg'].std():.2f}")
    print(f"最小值: {rent_price_df['gas_fee_avg'].min():.2f}")
    print(f"最大值: {rent_price_df['gas_fee_avg'].max():.2f}")
    print(f"中位数: {rent_price_df['gas_fee_avg'].median():.2f}")
    
    # 显示分布情况
    print(f"\n燃气费分布概况:")
    fee_stats = rent_price_df['gas_fee_avg'].describe()
    print(fee_stats)
    
    # 按区间显示分布
    bins = [0, 1, 2, 2.5, 3, 3.5, 4, 5, 10, float('inf')]
    labels = ['0-1', '1-2', '2-2.5', '2.5-3', '3-3.5', '3.5-4', '4-5', '5-10', '10+']
    fee_ranges = pd.cut(rent_price_df['gas_fee_avg'], bins=bins, labels=labels, right=False)
    print(f"\n燃气费区间分布:")
    print(fee_ranges.value_counts().sort_index())

print(f"缺失值数量: {rent_price_df['gas_fee_avg'].isna().sum()}")

print("燃气费变量创建完成")

燃气费分布:
燃气费
2.61元/m³        1285
3元/m³            924
3.45元/m³         861
3.5元/m³          647
1.98元/m³         305
                ... 
2.7-5元/m³          1
3.46元/m³           1
0.5-3.3元/m³        1
2.3-3.95元/m³       1
2.9-3.95元/m³       1
Name: count, Length: 188, dtype: int64

燃气费变量统计:
非缺失值数量: 6631
平均值: 2.91
标准差: 0.59
最小值: 0.40
最大值: 5.00
中位数: 2.96

燃气费分布概况:
count    6631.000000
mean        2.905798
std         0.592165
min         0.400000
25%         2.610000
50%         2.965000
75%         3.450000
max         5.000000
Name: gas_fee_avg, dtype: float64

燃气费区间分布:
gas_fee_avg
0-1         1
1-2       773
2-2.5     662
2.5-3    1895
3-3.5    2192
3.5-4     914
4-5       184
5-10       10
10+         0
Name: count, dtype: int64
缺失值数量: 3142
燃气费变量创建完成


In [22]:
# 2.21 供热费
# 2.21.1 分布检查
print("供热费分布:")
print(rent_price_df['供热费'].value_counts())
# 在初步调研时需要显示所有行，可以运行以下代码
# import pandas as pd
# pd.set_option('display.max_rows', None)
# print(rent_price_df['供热费'].value_counts())
# pd.reset_option('display.max_rows')

# 2.21.2 创建供热费变量
# 删除已存在的供热费变量列
heating_columns_to_drop = [col for col in rent_price_df.columns if 
                          col in ['heating_fee_avg']]
rent_price_df = rent_price_df.drop(columns=heating_columns_to_drop)

# 初始化供热费变量为NaN
rent_price_df['heating_fee_avg'] = np.nan

def extract_heating_fee(heating_fee_str):
    """
    从供热费字符串中提取平均值
    处理逻辑:
    - 单个数值: "30元/㎡" → 平均值=30
    - 区间数值: "25-35元/㎡" → 平均值=(25+35)/2=30
    - 单位统一为: 元/㎡
    """
    if pd.isna(heating_fee_str):
        return np.nan
    
    heating_fee_str = str(heating_fee_str).strip()
    
    try:
        # 处理单个数值格式
        if '元/㎡' in heating_fee_str and '-' not in heating_fee_str:
            # 提取数字部分
            fee_match = re.search(r'(\d+\.?\d*)', heating_fee_str)
            if fee_match:
                return float(fee_match.group(1))
        
        # 处理区间格式
        elif '-' in heating_fee_str and '元/㎡' in heating_fee_str:
            # 提取两个数字
            fee_range = re.findall(r'(\d+\.?\d*)', heating_fee_str)
            if len(fee_range) == 2:
                fee_min = float(fee_range[0])
                fee_max = float(fee_range[1])
                # 计算平均值
                return (fee_min + fee_max) / 2
        
        # 其他格式无法解析，返回NaN
        return np.nan
        
    except (ValueError, IndexError, AttributeError) as e:
        # 如果解析失败，返回NaN
        print(f"警告: 无法解析供热费 '{heating_fee_str}': {e}")
        return np.nan

# 应用转换函数
for idx in rent_price_df.index:
    heating_fee_str = rent_price_df.loc[idx, '供热费']
    if pd.notna(heating_fee_str):
        heating_fee_avg = extract_heating_fee(heating_fee_str)
        rent_price_df.loc[idx, 'heating_fee_avg'] = heating_fee_avg

print(f"\n供热费变量统计:")
print(f"非缺失值数量: {rent_price_df['heating_fee_avg'].notna().sum()}")
if rent_price_df['heating_fee_avg'].notna().sum() > 0:
    print(f"平均值: {rent_price_df['heating_fee_avg'].mean():.2f}")
    print(f"标准差: {rent_price_df['heating_fee_avg'].std():.2f}")
    print(f"最小值: {rent_price_df['heating_fee_avg'].min():.2f}")
    print(f"最大值: {rent_price_df['heating_fee_avg'].max():.2f}")
    print(f"中位数: {rent_price_df['heating_fee_avg'].median():.2f}")
    
    # 显示分布情况
    print(f"\n供热费分布概况:")
    fee_stats = rent_price_df['heating_fee_avg'].describe()
    print(fee_stats)
    
    # 按区间显示分布
    bins = [0, 1, 5, 10, 20, 25, 30, 35, 40, 50, float('inf')]
    labels = ['0-1', '1-5', '5-10', '10-20', '20-25', '25-30', '30-35', '35-40', '40-50', '50+']
    fee_ranges = pd.cut(rent_price_df['heating_fee_avg'], bins=bins, labels=labels, right=False)
    print(f"\n供热费区间分布:")
    print(fee_ranges.value_counts().sort_index())

print(f"缺失值数量: {rent_price_df['heating_fee_avg'].isna().sum()}")


print("供热费变量创建完成")

供热费分布:
供热费
30元/㎡           814
1元/㎡            339
24元/㎡           165
24-30元/㎡        154
22元/㎡           154
               ... 
4.91元/㎡           1
10元/㎡             1
0.1-1元/㎡          1
2.58-5.88元/㎡      1
29.9-30元/㎡        1
Name: count, Length: 98, dtype: int64

供热费变量统计:
非缺失值数量: 2719
平均值: 19.77
标准差: 12.79
最小值: 0.01
最大值: 45.00
中位数: 25.00

供热费分布概况:
count    2719.000000
mean       19.774989
std        12.787159
min         0.010000
25%         3.450000
50%        25.000000
75%        30.000000
max        45.000000
Name: heating_fee_avg, dtype: float64

供热费区间分布:
heating_fee_avg
0-1       81
1-5      672
5-10     122
10-20     72
20-25    407
25-30    298
30-35    873
35-40    144
40-50     50
50+        0
Name: count, dtype: int64
缺失值数量: 7054
供热费变量创建完成


In [23]:
# 2.22 停车位
# 2.22.1 分布检查
print("停车位分布:")
print(rent_price_df['停车位'].value_counts())

# 2.22.2 缺失值检查
print(f"\n停车位缺失值数量: {rent_price_df['停车位'].isna().sum()}")

# 2.22.3 创建停车位变量
# 删除已存在的停车位变量列
parking_columns_to_drop = [col for col in rent_price_df.columns if 
                          col in ['parking_spots']]
rent_price_df = rent_price_df.drop(columns=parking_columns_to_drop)

# 初始化停车位变量，直接使用原数据
rent_price_df['parking_spots'] = rent_price_df['停车位']

# 确保停车位变量为数值类型
rent_price_df['parking_spots'] = pd.to_numeric(rent_price_df['parking_spots'], errors='coerce')

print(f"\n停车位变量统计:")
print(f"非缺失值数量: {rent_price_df['parking_spots'].notna().sum()}")
if rent_price_df['parking_spots'].notna().sum() > 0:
    print(f"平均值: {rent_price_df['parking_spots'].mean():.2f}")
    print(f"标准差: {rent_price_df['parking_spots'].std():.2f}")
    print(f"最小值: {rent_price_df['parking_spots'].min():.2f}")
    print(f"最大值: {rent_price_df['parking_spots'].max():.2f}")
    print(f"中位数: {rent_price_df['parking_spots'].median():.2f}")
    
    # 显示分布情况
    print(f"\n停车位分布概况:")
    spots_stats = rent_price_df['parking_spots'].describe()
    print(spots_stats)
    
    # 按区间显示分布
    bins = [0, 1, 10, 50, 100, 200, 500, 1000, float('inf')]
    labels = ['0-1', '1-10', '10-50', '50-100', '100-200', '200-500', '500-1000', '1000+']
    spots_ranges = pd.cut(rent_price_df['parking_spots'], bins=bins, labels=labels, right=False)
    print(f"\n停车位区间分布:")
    print(spots_ranges.value_counts().sort_index())

print(f"缺失值数量: {rent_price_df['parking_spots'].isna().sum()}")

print("停车位变量创建完成")

停车位分布:
停车位
500.0     252
200.0     222
300.0     215
1000.0    161
100.0     134
         ... 
868.0       1
483.0       1
8.0         1
13.0        1
496.0       1
Name: count, Length: 646, dtype: int64

停车位缺失值数量: 3115

停车位变量统计:
非缺失值数量: 6658
平均值: 1229.52
标准差: 1538.58
最小值: 1.00
最大值: 8700.00
中位数: 700.00

停车位分布概况:
count    6658.000000
mean     1229.518023
std      1538.576934
min         1.000000
25%       300.000000
50%       700.000000
75%      1500.000000
max      8700.000000
Name: parking_spots, dtype: float64

停车位区间分布:
parking_spots
0-1            0
1-10          56
10-50        160
50-100       294
100-200      542
200-500     1448
500-1000    1402
1000+       2756
Name: count, dtype: int64
缺失值数量: 3115
停车位变量创建完成


In [24]:
# 2.21 到市中心距离
# Haversine公式计算两个经纬度点之间的距离（单位：公里）
def calculate_distance(lon1, lat1, lon2, lat2):
    # 将十进制度数转化为弧度
    lon1, lat1, lon2, lat2 = map(radians, [lon1, lat1, lon2, lat2])
    
    # Haversine公式
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    c = 2 * atan2(sqrt(a), sqrt(1-a))
    r = 6371  # 地球平均半径，单位为公里
    return c * r

# 定义各城市中心点经纬度字典
city_centers = {
    0: (116.3333, 39.9333),    # 城市0：北京
    1: (116.3333, 39.9333),    # 城市1：北京
    2: (106.33, 29.35),        # 城市2：重庆
    3: None,                   # 城市3：需要从数据中计算中心点
    4: (121.48, 31.23),        # 城市4：上海
    5: (113.56, 22.28),        # 城市5：广州
    6: (114.86, 40.8),         # 城市6：张家口
    7: (114.07, 22.62),        # 城市7：深圳（修正了经度）
    8: (102.7086, 25.036),     # 城市8：昆明
    9: None,                   # 城市9：需要从数据中计算中心点
    10: (113.2647, 23.1089),   # 城市10：佛山
    11: (115.4667, 38.8667)    # 城市11：保定
}

# 对于城市3和城市9，从数据中计算中心点（取该城市经纬度的平均值）
def calculate_city_center(df, city_id):
    city_data = df[df['城市'] == city_id]
    if len(city_data) == 0:
        return None
    center_lon = city_data['lon'].mean()
    center_lat = city_data['lat'].mean()
    return (center_lon, center_lat)

# 计算距市中心距离的函数
def calculate_city_center_distance(row):
    city_id = row['城市']
    center_point = city_centers.get(city_id)
    
    # 如果中心点未定义，则从数据中计算
    if center_point is None:
        center_point = calculate_city_center(rent_price_df, city_id)
        if center_point is None:
            return np.nan
    
    center_lon, center_lat = center_point
    distance = calculate_distance(row['lon'], row['lat'], center_lon, center_lat)
    return distance

# 应用函数计算距离
print("开始计算距市中心距离...")
rent_price_df['距市中心距离_km'] = rent_price_df.apply(calculate_city_center_distance, axis=1)

# 检查结果
print("\n=== 距市中心距离计算完成 ===")
print(f"成功计算了 {rent_price_df['距市中心距离_km'].notna().sum()} 条数据的距离")
print(f"缺失值数量: {rent_price_df['距市中心距离_km'].isna().sum()}")

# 显示各城市距离的统计信息
print("\n=== 各城市距市中心距离统计（公里）===")
distance_stats = rent_price_df.groupby('城市')['距市中心距离_km'].agg(['count', 'mean', 'min', 'max']).round(2)
print(distance_stats)

# 显示前几行数据，包含新计算的距离列
print("\n=== 包含距市中心距离的前5行数据 ===")
columns_to_show = ['城市', 'lon', 'lat', '距市中心距离_km']
if 'Price' in rent_price_df.columns:
    columns_to_show.insert(3, 'Price')
print(rent_price_df[columns_to_show].head())

print("\n处理完成！")

开始计算距市中心距离...

=== 距市中心距离计算完成 ===
成功计算了 9773 条数据的距离
缺失值数量: 0

=== 各城市距市中心距离统计（公里）===
    count    mean     min     max
城市                               
0    1512  148.21  105.19  210.52
1     567  147.89   95.38  181.47
2     936  181.25  152.47  241.85
3    1802   20.23    1.25   67.91
4    1168  145.91  103.79  183.26
5     592  142.47  107.66  164.82
6     188  146.12  137.43  260.93
7     837  151.06  133.22  179.77
8     668  152.80  123.94  173.09
9     340    3.00    0.21   32.52
10   1070  164.41  144.85  212.69
11     93  179.91  142.15  232.82

=== 包含距市中心距离的前5行数据 ===
   城市         lon        lat   距市中心距离_km
0   1  117.345687  40.447235  103.250283
1  10  114.279469  24.158183  155.879998
2   3  121.684578  32.198660   30.278446
3   0  117.323901  40.935338  139.439411
4   3  121.718296  32.335562   19.537753

处理完成！


In [25]:
# 2.27 物业办公电话
# 2.27.1 分布检查
print("物业办公电话分布:")
print(rent_price_df['物业办公电话'].value_counts().head(10))

# 2.27.2 异常值处理
# 检查是否有异常值或缺失值
print(f"\n物业办公电话缺失值数量: {rent_price_df['物业办公电话'].isna().sum()}")

# 2.27.3 创建物业办公电话哑变量
# 删除已存在的物业电话哑变量列
property_phone_columns_to_drop = [col for col in rent_price_df.columns if col.startswith('property_phone_')]
rent_price_df = rent_price_df.drop(columns=property_phone_columns_to_drop)

# 创建物业办公电话哑变量
# 规则：包含数字记为1，空值、"无"或非数字内容记为0
def has_phone_number(value):
    if pd.isna(value):
        return 0
    value_str = str(value).strip()
    if value_str == '' or value_str == '无' or value_str == '暂无' or value_str == '没有':
        return 0
    # 检查是否包含数字
    if any(char.isdigit() for char in value_str):
        return 1
    else:
        return 0

# 应用函数创建哑变量
rent_price_df['property_phone_yes'] = rent_price_df['物业办公电话'].apply(has_phone_number)

print(f"\n已创建的物业办公电话哑变量:")
print(f"property_phone_yes: 1表示有物业办公电话，0表示无物业办公电话")

print("\n物业办公电话哑变量分布:")
print(f"无物业办公电话 (property_phone_yes=0): {(rent_price_df['property_phone_yes'] == 0).sum()}")
print(f"有物业办公电话 (property_phone_yes=1): {(rent_price_df['property_phone_yes'] == 1).sum()}")
print(f"缺失值: {rent_price_df['property_phone_yes'].isna().sum()}")

print("物业办公电话哑变量创建完成")

物业办公电话分布:
物业办公电话
无                  346
暂无                  60
0316-3302980        56
0316-5996892        37
8816028/8816038     32
15801295270         30
0755-26084409       28
0755-86951939       28
010-58263311        25
010-58610780        23
Name: count, dtype: int64

物业办公电话缺失值数量: 6453

已创建的物业办公电话哑变量:
property_phone_yes: 1表示有物业办公电话，0表示无物业办公电话

物业办公电话哑变量分布:
无物业办公电话 (property_phone_yes=0): 6869
有物业办公电话 (property_phone_yes=1): 2904
缺失值: 0
物业办公电话哑变量创建完成


In [26]:
# 创建最终的数据集并保存到本地

# 列出所有需要的变量
final_variables = [
    'ID',
    'decoration',
    'high_dummy',
    'middle_dummy', 
    'low_dummy',
    'basement_dummy',
    'total_floor',
    'area',
    'room_count',
    'hall_count',
    'south_dummy',
    'north_south_dummy',
    'pay_annual',
    'pay_bi_monthly',
    'pay_monthly',
    'pay_quarterly',
    'pay_semi_annual',
    'rent_type_shared',
    'elevator_yes',
    'water_civil',
    'water_commercial',
    'electricity_civil',
    'electricity_commercial',
    'gas_yes',
    'heating_self',
    'lease_min_months',
    'lease_max_months',
    'lease_avg_months',
    'facility_洗衣机',
    'facility_空调',
    'facility_衣柜',
    'facility_电视',
    'facility_冰箱',
    'facility_热水器',
    'facility_床',
    'facility_天然气',
    'facility_暖气',
    'facility_宽带',
    'building_age',
    'household_total',
    'building_total',
    'greening_rate',
    'plot_ratio',
    'property_fee_avg',
    'structure_tower',
    'structure_slab',
    'structure_combined',
    'structure_bungalow',
    'gas_fee_avg',
    'heating_fee_avg',
    'parking_spots',
    '城市',  # 添加城市列用于分组回归
    '区县',
    '板块',
    '距市中心距离_km',
    'property_phone_yes'
]

# 检查哪些变量在数据集中存在
existing_variables = [var for var in final_variables if var in rent_price_df.columns]
missing_variables = [var for var in final_variables if var not in rent_price_df.columns]

print("数据集变量检查:")
print(f"存在的变量数量: {len(existing_variables)}")
print(f"缺失的变量数量: {len(missing_variables)}")

if missing_variables:
    print("\n缺失的变量:")
    for var in missing_variables:
        print(f"  - {var}")

# 创建最终数据集
final_df = rent_price_df[existing_variables].copy()

print(f"\n最终数据集形状: {final_df.shape}")
print(f"样本数量: {final_df.shape[0]}")
print(f"变量数量: {final_df.shape[1]}")

# 检查缺失值情况
print("\n各变量缺失值统计:")
missing_stats = final_df.isnull().sum()
missing_stats = missing_stats[missing_stats > 0].sort_values(ascending=False)

if len(missing_stats) > 0:
    print("有缺失值的变量:")
    for var, count in missing_stats.items():
        percentage = (count / len(final_df)) * 100
        print(f"  {var}: {count}个缺失值 ({percentage:.2f}%)")
else:
    print("没有缺失值")

# 保存到本地
output_file = 'rent_price_final_test_dataset.csv'
final_df.to_csv(output_file, index=False, encoding='utf-8-sig')

print(f"\n最终数据集已保存到: {output_file}")

# 显示数据集的基本信息
print("\n数据集基本信息:")
print(final_df.info())

# 显示前几行数据
print("\n数据集前5行:")
print(final_df.head())

# 显示城市分布（用于分组回归）
print("\n城市分布（用于分组回归）:")
print(final_df['城市'].value_counts())

数据集变量检查:
存在的变量数量: 56
缺失的变量数量: 0

最终数据集形状: (9773, 56)
样本数量: 9773
变量数量: 56

各变量缺失值统计:
有缺失值的变量:
  heating_fee_avg: 7054个缺失值 (72.18%)
  heating_self: 5974个缺失值 (61.13%)
  lease_max_months: 5175个缺失值 (52.95%)
  lease_avg_months: 5175个缺失值 (52.95%)
  lease_min_months: 5175个缺失值 (52.95%)
  facility_冰箱: 3694个缺失值 (37.80%)
  facility_天然气: 3694个缺失值 (37.80%)
  facility_暖气: 3694个缺失值 (37.80%)
  facility_宽带: 3694个缺失值 (37.80%)
  facility_床: 3694个缺失值 (37.80%)
  facility_洗衣机: 3694个缺失值 (37.80%)
  facility_衣柜: 3694个缺失值 (37.80%)
  facility_电视: 3694个缺失值 (37.80%)
  facility_热水器: 3694个缺失值 (37.80%)
  facility_空调: 3694个缺失值 (37.80%)
  building_age: 3381个缺失值 (34.60%)
  gas_fee_avg: 3142个缺失值 (32.15%)
  parking_spots: 3115个缺失值 (31.87%)
  greening_rate: 3019个缺失值 (30.89%)
  plot_ratio: 2988个缺失值 (30.57%)
  property_fee_avg: 2804个缺失值 (28.69%)
  pay_annual: 2387个缺失值 (24.42%)
  pay_quarterly: 2387个缺失值 (24.42%)
  pay_bi_monthly: 2387个缺失值 (24.42%)
  pay_monthly: 2387个缺失值 (24.42%)
  pay_semi_annual: 2387个缺失值 (24.42%)
  water_ci